<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.it/cap08/cap08.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

(Presumo que o texto a seguir seja o conteúdo a ser traduzido; porém, o texto em si não foi fornecido no prompt. Fornecendo-se o texto em Markdown, ele será traduzido conforme as regras acima.)

## 💻 **Parte Pratica con Esercizi di Programmazione**

La presente lista di esercizi di programmazione (EP) consolida le formulazioni teoriche presentate nel Capitolo 8 — Corrispondenza delle Caratteristiche, Rilevamento degli Oggetti e Segmentazione Classica — attraverso un percorso pratico applicato. Come nel capitolo precedente, gli EP isolano le **grandezze intermedie** di ciascuna tecnica — la distanza tra descrittori binari, i termini di un'immagine integrale, il conteggio degli *inlier* di un modello candidato, la sovrapposizione tra bounding box e l'etichetta di ciascuna componente connessa — consentendo di validare manualmente ogni fase del ragionamento senza dipendere da OpenCV né da immagini esterne.

La concatenazione degli esercizi riproduce il flusso concettuale del capitolo: si inizia con la **distanza di Hamming**, cuore della corrispondenza dei descrittori binari come ORB; si prosegue con il conteggio degli ***inlier*** che sostiene **RANSAC** nella stima robusta di un'omografia; si continua con l'**immagine integrale**, l'escamotage computazionale che rende l'***Haar Cascade*** praticabile in tempo reale; si approfondisce con **IoU e Soppressione Non-Massima**, il post-processamento comune a praticamente ogni rilevatore di oggetti; e si conclude con l'**etichettatura delle componenti connesse**, l'approccio classico — e le sue limitazioni — per segmentare istanze individuali in una maschera binaria.

### 🎯 Obiettivo di questo Quaderno

Il quaderno consente di sviluppare, validare, organizzare e testare soluzioni di **Esercizi di Programmazione (EPs)** in ambienti interattivi, come Colab, con gli stessi casi di test di Moodle, copiandoli lì solo al momento di registrare il voto ufficiale.

### *Download*

Scarica `morph.py` e `testsuite.py` eseguendo la cella qui sotto:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Esecuzione dei Test
Per valutare i test, esegui `TestSuite("EP08_01.estensione").run()` in una nuova cella, sostituendo l'estensione con quella del linguaggio utilizzato (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). Il sistema scarica i casi di test da GitHub, esegue il programma e calcola automaticamente il voto.

Per testare il codice Python direttamente, senza salvare un file, usa `run_code(codice)` passando il codice come *stringa* in una variabile `codice`:

```python
codice = """
# ... il tuo codice qui ...
"""
TestSuite("EP08_01").run_code(codice)
```

### 🛠️ Riepilogo dei Metodi di `morph.py` (Cap. 8)

La libreria `morph.py` fornisce funzioni per l'analisi dei componenti connessi, l'estrazione dei contorni, le metriche geometriche e le annotazioni:

1. **Componenti e Contorni (`connectedComponents`, `findContours`)**
Etichettano le regioni connesse ed estraggono i contorni di immagini binarie.
2. **Proprietà del Contorno (`contourArea`, `arcLength`, `convexHull`, `approxPolyDP`, `fitLine`)**
Calcolano area, perimetro, inviluppo convesso, approssimazione poligonale e adattamento di una retta per un contorno.
3. **Geometria di Involucro (`boundingRect`, `minAreaRect`, `boxPoints`, `minEnclosingCircle`, `fitEllipse`)**
Determinano rettangoli di delimitazione (allineati o orientati), ellissi e il cerchio minimo circoscritto.
4. **Estrazione e Salvataggio delle Misure (`measure`, `saveMeasures`)**
Estraggano descrittori geometrici degli oggetti (area, circolarità, solidità, centroide) ed esportano i dati in formato CSV, testo o YOLO.
5. **Valutazione e Visualizzazione (`IoU`, `verifyBoundBox`, `showBoundBox`)**
Calcolano l'intersezione su unione (*Intersection over Union*), validano i bounding box con le ground truth e disegnano caselle di delimitazione annotate sull'immagine.

### EP08_01 🟢 Distanza di Hamming e Corrispondenza di Descrittori Binari

L’ORB, utilizzato nel Progetto Pratico 1 di questo capitolo, descrive l’intorno di ogni punto di interesse come una sequenza di bit — e, per questo, il confronto tra due descrittori non usa la distanza euclidea del k-NN del Capitolo 7, bensì la **distanza di Hamming**: il numero di posizioni in cui i bit differiscono. Prima di chiamare `cv2.BFMatcher(cv2.NORM_HAMMING)`, ti è stato richiesto di implementare manualmente questa corrispondenza (*matching*) per forza bruta — la stessa fase che, eseguita internamente da OpenCV, precede la stima robusta dell’omografia tramite RANSAC.

#### 📋 Linee Guida di Implementazione

1. **Quantità:** Leggere gli interi $N$ e $M$ — numero di descrittori estratti dall’immagine A e dall’immagine B, rispettivamente.
2. **Descrittori di A:** Leggere $N$ righe, ciascuna contenente un descrittore binario (una *stringa* di caratteri `0` e `1`, tutti della stessa lunghezza).
3. **Descrittori di B:** Leggere $M$ righe, nello stesso formato.
4. **Soglia:** Leggere l’intero $\tau$ — distanza di Hamming massima accettabile per considerare valida una corrispondenza.
5. **Distanza di Hamming:** Per due descrittori binari $a$ e $b$ della stessa lunghezza,
$$
d_H(a, b) = \sum_{k} \mathbb{1}[a_k \neq b_k],
$$
   cioè il conteggio delle posizioni in cui i bit differiscono.
6. **Corrispondenza per vicino più prossimo:** Per ogni descrittore $a_i$ di A ($i$ nell’ordine di lettura, a partire da $0$), calcola la sua distanza di Hamming da **tutti** i descrittori di B e trova quello con la distanza minima. In caso di parità tra due o più descrittori di B con la stessa distanza minima, scegli quello con **indice minore**.
7. **Filtraggio tramite soglia:** Se la distanza minima trovata è $\le \tau$, la corrispondenza è valida; altrimenti, $a_i$ non ha corrispondenza.
8. **Output:** Per ogni $i$ da $0$ a $N-1$, nell’ordine di lettura, stampare una riga: `i j d` se esiste una corrispondenza valida (dove $j$ è l’indice del descrittore di B scelto e $d$ la sua distanza), oppure `i -1` in caso contrario. Alla fine, stampare `Total correspondências válidas: X`.

#### 📌 Vincoli Computazionali

* **Stessa lunghezza:** tutti i descrittori (di A e di B) hanno esattamente lo stesso numero di bit.
* **Forza bruta:** confronta ogni descrittore di A con **tutti** quelli di B — non è necessaria alcuna indicizzazione o struttura di accelerazione.
* **Pareggio risolto per indice minore in B**, e **mai** per ordine di lettura di A (che è già naturale, poiché ogni $a_i$ è trattato in modo indipendente).

#### 🧠 Fondamenti Teorici

| Elemento | Ruolo nella corrispondenza ORB |
|---|---|
| Descrittore binario (BRIEF) | Ogni bit è il risultato di un confronto di intensità tra due pixel dell’intorno |
| Distanza di Hamming | Metrica di dissimmetria tra *stringhe* binarie; molto più rapida da calcolare rispetto alla distanza euclidea (operazione XOR + conteggio dei bit) |
| Vicino più prossimo | Criterio di corrispondenza: ogni punto di A è accoppiato al punto di B con descrittore più simile |
| Soglia $\tau$ | Filtra corrispondenze poco affidabili ancor prima del RANSAC — ma, come discusso nel capitolo, alcune corrispondenze errate passano comunque, richiedendo la robustezza del RANSAC |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $N$ e $M$.
* Prossime $N$ righe: un descrittore binario per riga (*stringa* di `0` e `1`).
* Prossime $M$ righe: un descrittore binario per riga, nello stesso formato.
* Ultima riga: Intero $\tau$.

**Output:**

* $N$ righe, una per descrittore di A, nel formato `i j d` o `i -1`.
* Ultima riga: `Total correspondências válidas: X`.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 3 3<br>10101010<br>11110000<br>00001111<br>10101011<br>00001110<br>11111111<br>2 | 0 0 1<br>1 -1<br>2 1 1<br>Total correspondências válidas: 2 | Il descrittore `11110000` non trova corrispondenza: il suo vicino più prossimo è a distanza 4, superiore alla soglia $\tau=2$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0801" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0801 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0801 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0801 button:hover { background: #e8dfcf; }
  .sim-ep0801_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0801_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0801_bit { width: 36px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-family: monospace; font-weight: 700; font-size: 14px; user-select: none; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP08_01: Distanza di Hamming tra Descrittori Binari</span>
  <span class="sim-ep0801_pill">Descrittori a 8 Bit</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel Informativo de Instruções -->
  <div class="sim-ep0801_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center;">
      Clicca su qualsiasi bit del <b>Descrittore B</b> per invertirlo e osserva la distanza di Hamming cambiare in tempo reale.
    </div>
  </div>

  <!-- Grades dos Descritores -->
  <div class="sim-ep0801_panel" style="margin-bottom:14px; display:flex; flex-direction:column; gap:12px; align-items:center;">
    <div>
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:6px; text-align:center; letter-spacing:0.04em;">
        Descrittore A (Fisso)
      </div>
      <div id="sim-ep0801_a" style="display:grid; grid-template-columns:repeat(8, 36px); gap:4px;"></div>
    </div>

    <div>
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:6px; text-align:center; letter-spacing:0.04em;">
        Descrittore B (Clicca per Invertire)
      </div>
      <div id="sim-ep0801_b" style="display:grid; grid-template-columns:repeat(8, 36px); gap:4px;"></div>
    </div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0801_debug" class="sim-ep0801_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep01(root){
    if (!root || root.dataset.sim08Ep01Init) return;
    root.dataset.sim08Ep01Init = "1";

    var A = [1, 0, 1, 0, 1, 0, 1, 0];
    var B = [1, 0, 1, 0, 1, 0, 1, 1];

    var aEl = root.querySelector('#sim-ep0801_a');
    var bEl = root.querySelector('#sim-ep0801_b');
    var dbg = root.querySelector('#sim-ep0801_debug');

    function estiloBit(destacado, interativo){
      var base = destacado 
        ? 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;' 
        : 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;';
      var cursor = interativo ? ' cursor:pointer;' : ' cursor:default;';
      return base + cursor;
    }

    function render(){
      aEl.innerHTML = ''; 
      bEl.innerHTML = '';
      var dist = 0;

      for (var k = 0; k < 8; k++){
        var diff = A[k] !== B[k];
        if (diff) dist++;

        var da = document.createElement('div');
        da.className = 'sim-ep0801_bit';
        da.style.cssText = estiloBit(diff, false);
        da.textContent = A[k];
        aEl.appendChild(da);

        var db = document.createElement('div');
        db.className = 'sim-ep0801_bit';
        db.style.cssText = estiloBit(diff, true);
        db.textContent = B[k];
        
        (function(idx){
          db.addEventListener('click', function(){
            B[idx] = 1 - B[idx];
            render();
          });
        })(k);

        bEl.appendChild(db);
      }

      if (dist === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#26241d';
      }

      dbg.textContent = 'A = ' + A.join('') + '   B = ' + B.join('') + '   →   Distância de Hamming = ' + dist;
    }

    render();
  }

  function tryInitSim08Ep01(){
    var root = document.getElementById('sim-ep0801');
    if (root) initSim08Ep01(root); else setTimeout(tryInitSim08Ep01, 200);
  }
  tryInitSim08Ep01();
})();
</script>
""")

**Figura 8.1:** Simulatore EP08_01: Distanza di Hamming tra Due Descrittori Binari


In [ ]:
%%writefile EP08_01.py
# Codice Python

In [ ]:
TestSuite("EP08_01.py").run()

### EP08_02 🟢 Omografia e RANSAC: Il Voto per *Inlier*

Il RANSAC, presentato nella sezione "Modellazione Matematica: Omografia e RANSAC", ripete un ciclo di tre passaggi — selezionare un campione minimo, stimare un modello candidato e contare quante corrispondenze sono coerenti con esso (gli ***inlier***) — conservando alla fine il modello più votato. La fase di stima del modello a partire da 4 punti (passo 2) coinvolge algebra lineare che esula dallo scopo di questo EP; qui ricevi direttamente un insieme di omografie **già candidate** — come se ciascuna fosse stata stimata da un campione casuale diverso — e sei incaricato di riprodurre esattamente il passo decisivo dell'algoritmo: **applicare ogni modello a tutte le corrispondenze e contare i suoi *inlier***, scegliendo il vincitore.

#### 📋 Linee Guida di Implementazione

1. **Corrispondenze:** Leggere l'intero $N$ e, successivamente, $N$ righe con quattro reali ciascuna, $x\ y\ x'\ y'$ — un punto dell'immagine A e il suo corrispondente (possibilmente errato) nell'immagine B, esattamente come prodotto dalla fase di *matching* dell'EP08_01.
2. **Modelli candidati:** Leggere l'intero $K$ (numero di omografie candidate) e il reale $\varepsilon$ (soglia di errore di riproiezione). Successivamente, leggere $K$ righe, ciascuna con nove reali $h_{11}\ h_{12}\ h_{13}\ h_{21}\ h_{22}\ h_{23}\ h_{31}\ h_{32}\ h_{33}$ — gli elementi della matrice $H$ candidata, in ordine di lettura per righe (*row-major*).
3. **Riproiezione:** Per ogni corrispondenza $(x,y,x',y')$ e ogni modello candidato $H_k$, calcolare il punto proiettato
$$
\begin{bmatrix} \hat x \\ \hat y \\ \hat w \end{bmatrix} = H_k \begin{bmatrix} x \\ y \\ 1 \end{bmatrix},
\qquad
(\hat x / \hat w,\ \hat y / \hat w)\ \text{è il punto proiettato.}
$$
4. **Errore di riproiezione:** $e = \sqrt{(\hat x/\hat w - x')^2 + (\hat y /\hat w - y')^2}$.
5. **Conteggio degli *inlier*:** Una corrispondenza è un *inlier* del modello $H_k$ se $e \le \varepsilon$.
6. **Selezione del modello migliore:** Il modello vincitore è quello con il maggior numero di *inlier*; in caso di pareggio, scegli quello con **indice minore** $k$ (il primo trovato durante il ciclo iterativo del RANSAC).
7. **Output:** Per ogni modello $k$ da $0$ a $K-1$, nell'ordine di lettura, stampare `Modello k: I inliers`. Alla fine, stampare `Modello migliore: k_best con I_best inliers`.

#### 📌 Vincoli Computazionali

* **Confronto inclusivo:** un errore di riproiezione **esattamente uguale** a $\varepsilon$ conta come *inlier* ($e \le \varepsilon$).
* **Senza stima di $H$:** le matrici sono già fornite pronte — non è necessario (né previsto) risolvere alcun sistema lineare.
* **Pareggio risolto con l'indice minore**, riflettendo il comportamento naturale di un algoritmo iterativo che esamina i modelli in ordine e sostituisce il migliore trovato finora solo quando un nuovo modello lo **supera strettamente**.

#### 🧠 Fondamenti Teorici

| Elemento | Ruolo nel RANSAC |
|---|---|
| Campione minimo (4 coppie) | Sufficiente per determinare gli 8 gradi di libertà di un'omografia |
| Modello candidato $H_k$ | Stimato da un campione minimo; può essere buono o cattivo, a seconda che il campione contenesse *outlier* |
| Errore di riproiezione | Misura quanto bene il modello "prevede" ogni corrispondenza osservata |
| *Inlier* vs. *outlier* | Corrispondenze coerenti con il modello vincitore (*inlier*) vs. le altre, tipicamente corrispondenze errate del *matching* |
| Rifinitura finale | In pratica, dopo aver scelto il modello migliore, il RANSAC lo ricalcola usando **solo** i suoi *inlier* — passo non richiesto in questo EP |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $N$.
* Prossime $N$ righe: quattro reali $x\ y\ x'\ y'$.
* Riga successiva: Intero $K$ e reale $\varepsilon$.
* Prossime $K$ righe: nove reali (elementi di $H_k$, *row-major*).

**Output:**

* $K$ righe nel formato `Modello k: I inliers`.
* Ultima riga: `Modello migliore: k_best con I_best inliers`.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 5<br>0 0 0 0<br>1 1 2 2<br>2 0 4 0<br>0 2 0 4<br>5 5 1 1<br>2 0.5<br>2 0 0 0 2 0 0 0 1<br>1 0 0 0 1 0 0 0 1 | Modello 0: 4 inliers<br>Modello 1: 1 inliers<br>Modello migliore: 0 con 4 inliers | Il Modello 0 (scala ×2) spiega correttamente 4 delle 5 corrispondenze; la 5ª, $(5,5)\to(1,1)$, è un *outlier* che nessuno dei due modelli spiega bene. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0802" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0802 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0802 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0802 button:hover { background: #e8dfcf; }
  #sim-ep0802 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0802_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0802_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP08_02: RANSAC &mdash; Conteggio degli Inlier</span>
  <span class="sim-ep0802_pill">Modello: Scala &times;2</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0808_panel sim-ep0802_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Soglia (&epsilon;): <span id="sim-ep0802_vl" style="font-family:monospace; color:#26241d;">0.50</span>
      </label>
    </div>
    
    <input id="sim-ep0802_sl" type="range" min="0" max="13" step="0.25" value="0.5">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Il modello candidato mappa (x,y) &rarr; in (2x,2y). Regola la soglia &epsilon; e osserva quali corrispondenze diventano inlier o outlier.
    </div>
  </div>

  <!-- Cards de Pontos / Correspondências -->
  <div id="sim-ep0802_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:8px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0802_debug" class="sim-ep0802_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep02(root){
    if (!root || root.dataset.sim08Ep02Init) return;
    root.dataset.sim08Ep02Init = "1";

    var pontos = [
      {x:0, y:0, xp:0, yp:0},
      {x:1, y:1, xp:2, yp:2},
      {x:2, y:0, xp:4, yp:0},
      {x:0, y:2, xp:0, yp:4},
      {x:5, y:5, xp:1, yp:1}
    ];

    var slEl  = root.querySelector('#sim-ep0802_sl');
    var vlEl  = root.querySelector('#sim-ep0802_vl');
    var cards = root.querySelector('#sim-ep0802_cards');
    var dbg   = root.querySelector('#sim-ep0802_debug');

    function render(){
      var eps = parseFloat(slEl.value);
      vlEl.textContent = eps.toFixed(2);
      cards.innerHTML = '';
      var inliers = 0;

      pontos.forEach(function(p){
        var px = 2 * p.x, py = 2 * p.y;
        var erro = Math.sqrt((px - p.xp) * (px - p.xp) + (py - p.yp) * (py - p.yp));
        var dentro = erro <= eps;
        if (dentro) inliers++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
            : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">(' + p.x + ',' + p.y + ') &rarr; (' + p.xp + ',' + p.yp + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">erro = ' + erro.toFixed(2) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'INLIER' : 'outlier') + '</div>';

        cards.appendChild(div);
      });

      if (inliers > 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = '\u03B5 = ' + eps.toFixed(2) + '  |  inliers = ' + inliers + ' de ' + pontos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim08Ep02(){
    var root = document.getElementById('sim-ep0802');
    if (root) initSim08Ep02(root); else setTimeout(tryInitSim08Ep02, 200);
  }
  tryInitSim08Ep02();
})();
</script>
""")

**Figura 8.2:** Simulatore EP08_02: RANSAC — Votazione per Inlier tra Modelli Candidati


In [ ]:
%%writefile EP08_02.py
# Codice Python

In [ ]:
TestSuite("EP08_02.py").run()

### EP08_03 🟢 Immagine Integrale: Somme Rettangolari a Tempo Costante

Immagina una telecamera di sorveglianza che elabora 30 fotogrammi al secondo e, per ogni fotogramma, il sistema deve analizzare l'immagine in decine di posizioni e scale diverse, testando in ciascuna un insieme di caratteristiche rettangolari per decidere "c'è un volto qui?". Se calcolare la somma delle intensità di ogni rettangolo richiedesse sommare pixel per pixel, il sistema non avrebbe alcuna possibilità di valutare in tempo reale — il collo di bottiglia sarebbe proprio nella parte più ripetuta dell'algoritmo. È esattamente questo collo di bottiglia che l'immagine integrale elimina.

La Haar Cascade valuta migliaia di caratteristiche rettangolari per finestra, in molteplici posizioni e scale — qualcosa di irrealizzabile in tempo reale se ogni rettangolo richiedesse di sommare i propri pixel uno a uno. L'**immagine integrale**, definita nella sezione sulla Haar Cascade, risolve questo problema: una volta pre-calcolata, la somma delle intensità di **qualsiasi** regione rettangolare si ottiene con solo quattro query e tre operazioni aritmetiche, indipendentemente dalla dimensione del rettangolo.

Sei stato incaricato di implementare questa struttura da zero: in primo luogo, calcolare l'immagine integrale dall'immagine originale; successivamente, rispondere a query rettangolari arbitrarie.

#### 📋 Linee Guida di Implementazione

1. **Input:** Leggere le dimensioni $H \times W$ dell'immagine e i suoi $H \times W$ valori interi di intensità.
2. **Immagine integrale:** Calcolare, per ogni posizione $(i,j)$ (indicizzazione a partire da $0$, `[riga][colonna]`),
$$
II(i,j) = \sum_{i' \le i,\ j' \le j} I(i', j'),
$$
   ovvero la somma di tutti i pixel sopra e a sinistra di $(i,j)$, inclusa la posizione stessa.
3. **Query:** Leggere l'intero $Q$ e, successivamente, $Q$ righe, ciascuna con quattro interi $x_1\ y_1\ x_2\ y_2$ — gli angoli superiore-sinistro e inferiore-destro di un rettangolo, **entrambi inclusivi**, con $0 \le x_1 \le x_2 < W$ e $0 \le y_1 \le y_2 < H$.
4. **Somma rettangolare in O(1):** Per ogni query, calcolare la somma delle intensità all'interno del rettangolo utilizzando esclusivamente valori già presenti in $II$ (senza scorrere i pixel originali):
$$
S(x_1,y_1,x_2,y_2) = II(y_2,x_2) - II(y_2, x_1{-}1) - II(y_1{-}1, x_2) + II(y_1{-}1, x_1{-}1),
$$
   trattando qualsiasi termine con indice di riga o colonna uguale a $-1$ come $0$.
5. **Output:** In primo luogo, stampare l'immagine integrale completa — $H$ righe con $W$ interi ciascuna. Successivamente, per ogni query, stampare un singolo intero: la somma della regione corrispondente.

#### 📌 Vincoli Computazionali

* **Non ricalcolare per forza bruta:** la risposta a ogni query deve utilizzare la formula a quattro termini su $II$, non una somma diretta dei pixel del rettangolo (sebbene il risultato numerico sia lo stesso, l'obiettivo dell'esercizio è proprio questa tecnica).
* **Rettangoli con coordinate inclusive:** $(x_1,y_1)$ e $(x_2,y_2)$ appartengono alla regione sommata.
* **Gestione dei bordi:** quando si interroga $II$ con indice $-1$ (quando $x_1=0$ o $y_1=0$), utilizzare il valore $0$.

#### 🧠 Fondamentazione Teorica

| Elemento | Ruolo nella Haar Cascade |
|---|---|
| Immagine integrale $II$ | Pre-calcolata una sola volta per immagine, in tempo $O(HW)$ |
| Query in O(1) | Ogni caratteristica Haar (differenza tra somme di regioni rettangolari) viene valutata con poche operazioni, indipendentemente dall'area del rettangolo |
| Scalabilità | È questa costanza che rende possibile valutare migliaia di caratteristiche, in molteplici posizioni e scale, in tempo reale |
| Principio di inclusione-esclusione | I quattro termini della formula sommano la regione desiderata e sottraggono esattamente le aree conteggiate in eccesso |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $H$ e $W$.
* Prossime $H$ righe: $W$ interi ciascuna (immagine originale).
* Riga successiva: Intero $Q$.
* Prossime $Q$ righe: quattro interi $x_1\ y_1\ x_2\ y_2$.

**Output:**

* $H$ righe con $W$ interi ciascuna (l'immagine integrale).
* $Q$ righe, una per query, con la somma della regione corrispondente.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>1<br>0 0 2 2 | 1 3 6<br>5 12 21<br>12 27 45<br>45 | La query copre l'intera immagine; la somma coincide con $II(2,2)$ e con la somma di tutti i 9 valori. |
| 3 3<br>1 2 3<br>4 5 6<br>7 8 9<br>2<br>1 1 2 2<br>0 0 1 1 | 1 3 6<br>5 12 21<br>12 27 45<br>28<br>12 | La prima query usa i quattro termini della formula; la seconda coincide direttamente con $II(1,1)$, poiché inizia dall'origine. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0803" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0803 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0803 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0803 button:hover { background: #e8dfcf; }
  #sim-ep0803 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0803_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0803_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP08_03: Somma Rettangolare con Immagine Integrale</span>
  <span id="sim-ep0803_badge" class="sim-ep0803_pill">Interno</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição e Botões de Preset -->
  <div class="sim-ep0803_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Scegli un rettangolo (x<sub>1</sub>, y<sub>1</sub>) &ndash; (x<sub>2</sub>, y<sub>2</sub>). L'immagine integrale II include un bordo virtuale (&minus;1) con zeri per una validazione senza eccezioni.
    </div>

    <div style="display:flex; gap:6px; justify-content:center; flex-wrap:wrap;">
      <button data-preset="0,0,3,3">Dall'Origine</button>
      <button data-preset="0,1,2,3">Bordo Sinistro</button>
      <button data-preset="1,0,3,2">Bordo Superiore</button>
      <button data-preset="1,1,2,2">Completamente Interno</button>
      <button data-preset="2,2,2,2">Pixel Singolo</button>
    </div>
  </div>

  <!-- Sliders de Seleção das Coordenadas -->
  <div class="sim-ep0803_panel" style="margin-bottom:14px;">
    <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:12px;">
      
      <div>
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:4px;">
          Angolo Superiore-Sinistro (x<sub>1</sub>, y<sub>1</sub>) = <span id="sim-ep0803_v_tl" style="font-family:monospace; color:#26241d;">(1,1)</span>
        </div>
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600;">x<sub>1</sub></div>
        <input id="sim-ep0803_x1" type="range" min="0" max="3" step="1" value="1">
        
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600; margin-top:4px;">y<sub>1</sub></div>
        <input id="sim-ep0803_y1" type="range" min="0" max="3" step="1" value="1">
      </div>

      <div>
        <div style="font-size:11px; font-weight:700; color:#5e5a4a; margin-bottom:4px;">
          Angolo Inferiore-Destro (x<sub>2</sub>, y<sub>2</sub>) = <span id="sim-ep0803_v_br" style="font-family:monospace; color:#26241d;">(2,2)</span>
        </div>
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600;">x<sub>2</sub></div>
        <input id="sim-ep0803_x2" type="range" min="0" max="3" step="1" value="2">
        
        <div style="display:flex; justify-content:space-between; align-items:center; font-size:10px; color:#8a8371; font-weight:600; margin-top:4px;">y<sub>2</sub></div>
        <input id="sim-ep0803_y2" type="range" min="0" max="3" step="1" value="2">
      </div>

    </div>
  </div>

  <!-- Grades das Matrizes -->
  <div style="display:flex; gap:20px; justify-content:center; flex-wrap:wrap; margin-bottom:14px;">
    <div class="sim-ep0803_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:8px; letter-spacing:0.04em;">
        Immagine Originale I (4&times;4)
      </div>
      <div id="sim-ep0803_gridI" style="display:grid; justify-content:center;"></div>
    </div>

    <div class="sim-ep0803_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:8px; letter-spacing:0.04em;">
        Immagine Integrale II (Con Bordo Virtuale &minus;1)
      </div>
      <div id="sim-ep0803_gridII" style="display:grid; justify-content:center;"></div>
    </div>
  </div>

  <!-- Legenda das Operações -->
  <div style="display:flex; gap:12px; justify-content:center; flex-wrap:wrap; margin-bottom:14px; font-size:10px; font-weight:700; color:#5e5a4a;">
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#2980b9; border-radius:2px; display:inline-block;"></span> + II(y<sub>2</sub>, x<sub>2</sub>)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#d35400; border-radius:2px; display:inline-block;"></span> &minus; II(y<sub>2</sub>, x<sub>1</sub>&minus;1)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#d35400; border-radius:2px; display:inline-block;"></span> &minus; II(y<sub>1</sub>&minus;1, x<sub>2</sub>)</span>
    <span style="display:flex; align-items:center; gap:4px;"><span style="width:10px; height:10px; background:#27ae60; border-radius:2px; display:inline-block;"></span> + II(y<sub>1</sub>&minus;1, x<sub>1</sub>&minus;1)</span>
  </div>

  <!-- Painéis Informativos / Resultados -->
  <div id="sim-ep0803_formula" class="sim-ep0803_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center; margin-bottom:8px;"></div>
  <div id="sim-ep0803_verify" class="sim-ep0803_panel" style="font-family:monospace; font-size:11px; color:#04342C; background:#eafaf1; border-color:#a3e4d7; text-align:center;"></div>

</div>
</div>

<script>
(function(){
  function initSim08Ep03(root){
    if (!root || root.dataset.sim08Ep03Init) return;
    root.dataset.sim08Ep03Init = "1";

    var I = [
      [2, 1, 3, 4],
      [5, 6, 1, 2],
      [3, 2, 4, 1],
      [1, 3, 2, 5]
    ];
    var N = 4;
    var II = [];
    for (var i = 0; i < N; i++){ II.push([0, 0, 0, 0]); }
    
    for (var i = 0; i < N; i++){
      for (var j = 0; j < N; j++){
        II[i][j] = I[i][j] +
          (i > 0 ? II[i - 1][j] : 0) + 
          (j > 0 ? II[i][j - 1] : 0) - 
          (i > 0 && j > 0 ? II[i - 1][j - 1] : 0);
      }
    }

    var x1El     = root.querySelector('#sim-ep0803_x1');
    var y1El     = root.querySelector('#sim-ep0803_y1');
    var x2El     = root.querySelector('#sim-ep0803_x2');
    var y2El     = root.querySelector('#sim-ep0803_y2');
    var vTl      = root.querySelector('#sim-ep0803_v_tl');
    var vBr      = root.querySelector('#sim-ep0803_v_br');
    var gridI    = root.querySelector('#sim-ep0803_gridI');
    var gridII   = root.querySelector('#sim-ep0803_gridII');
    var formulaEl= root.querySelector('#sim-ep0803_formula');
    var verifyEl = root.querySelector('#sim-ep0803_verify');
    var badge    = root.querySelector('#sim-ep0803_badge');

    var CELL = 34, HEAD = 20;

    function cellDiv(text, size, extraStyle){
      var d = document.createElement('div');
      d.style.cssText = 'display:flex; align-items:center; justify-content:center; font-family:monospace; font-size:' + size + 'px;' + extraStyle;
      d.textContent = text;
      return d;
    }

    function clampAndRender(changed){
      var x1 = +x1El.value, y1 = +y1El.value, x2 = +x2El.value, y2 = +y2El.value;
      if (changed === 'x1' && x1 > x2) x2El.value = x1;
      if (changed === 'x2' && x2 < x1) x1El.value = x2;
      if (changed === 'y1' && y1 > y2) y2El.value = y1;
      if (changed === 'y2' && y2 < y1) y1El.value = y2;
      render();
    }

    function render(){
      var x1 = +x1El.value, y1 = +y1El.value, x2 = +x2El.value, y2 = +y2El.value;
      vTl.textContent = '(' + x1 + ',' + y1 + ')';
      vBr.textContent = '(' + x2 + ',' + y2 + ')';

      var sit;
      if (x1 === x2 && y1 === y2) sit = 'Pixel Único';
      else if (x1 === 0 && y1 === 0) sit = 'Desde a Origem';
      else if (x1 === 0) sit = 'Borda Esquerda';
      else if (y1 === 0) sit = 'Borda Superior';
      else sit = 'Interno';

      badge.textContent = sit;

      // Grade I
      gridI.style.gridTemplateColumns = HEAD + 'px repeat(' + N + ',' + CELL + 'px)';
      gridI.style.gridTemplateRows = HEAD + 'px repeat(' + N + ',' + CELL + 'px)';
      gridI.innerHTML = '';
      gridI.appendChild(cellDiv('', 10, 'color:#8a8371;'));
      for (var c = 0; c < N; c++) gridI.appendChild(cellDiv(c, 10, 'color:#8a8371; font-weight:700;'));
      
      for (var r = 0; r < N; r++){
        gridI.appendChild(cellDiv(r, 10, 'color:#8a8371; font-weight:700;'));
        for (var c = 0; c < N; c++){
          var dentro = (r >= y1 && r <= y2 && c >= x1 && c <= x2);
          gridI.appendChild(cellDiv(I[r][c], 12, 'border-radius:4px; transition:all 0.15s ease;' +
            (dentro 
              ? 'background:#f1ead7; border:2px solid #26241d; font-weight:700; color:#26241d;'
              : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;')));
        }
      }

      // Grade II (5x5 dados)
      var M = N + 1;
      gridII.style.gridTemplateColumns = HEAD + 'px repeat(' + M + ',' + CELL + 'px)';
      gridII.style.gridTemplateRows = HEAD + 'px repeat(' + M + ',' + CELL + 'px)';
      gridII.innerHTML = '';
      gridII.appendChild(cellDiv('', 10, 'color:#8a8371;'));
      for (var c2 = 0; c2 < M; c2++) gridII.appendChild(cellDiv(c2 - 1, 10, 'color:#8a8371; font-weight:700;'));

      var t1 = [y2 + 1, x2 + 1];
      var t2 = [y2 + 1, x1];
      var t3 = [y1, x2 + 1];
      var t4 = [y1, x1];

      function styleFor(r2, c2){
        var isVirtual = (r2 === 0 || c2 === 0);
        var base = isVirtual
          ? 'border-radius:4px; background:#fafaf7; border:1px dashed #e4dcc8; color:#8a8371;'
          : 'border-radius:4px; background:#fafaf7; border:1px solid #e4dcc8; color:#26241d;';
        
        function match(t, color, tcolor){
          if (r2 === t[0] && c2 === t[1]) {
            return 'border-radius:4px; font-weight:700; background:' + color + '; border:2px solid ' + tcolor + '; color:#ffffff;';
          }
          return null;
        }

        return match(t1, '#2980b9', '#1c5d85') || 
               match(t2, '#d35400', '#a04000') ||
               match(t3, '#d35400', '#a04000') || 
               match(t4, '#27ae60', '#1e8449') || base;
      }

      for (var r2 = 0; r2 < M; r2++){
        gridII.appendChild(cellDiv(r2 - 1, 10, 'color:#8a8371; font-weight:700;'));
        for (var c2 = 0; c2 < M; c2++){
          var val = (r2 === 0 || c2 === 0) ? 0 : II[r2 - 1][c2 - 1];
          gridII.appendChild(cellDiv(val, 12, styleFor(r2, c2)));
        }
      }

      function term(y, x){ return (y < 0 || x < 0) ? 0 : II[y][x]; }
      var a = term(y2, x2), b = term(y2, x1 - 1), c3 = term(y1 - 1, x2), d = term(y1 - 1, x1 - 1);
      var S = a - b - c3 + d;

      formulaEl.innerHTML =
        'S = II(' + y2 + ',' + x2 + ') &minus; II(' + y2 + ',' + (x1 - 1) + ') &minus; II(' + (y1 - 1) + ',' + x2 + ') + II(' + (y1 - 1) + ',' + (x1 - 1) + ')<br>' +
        'S = ' + a + ' &minus; ' + b + ' &minus; ' + c3 + ' + ' + d + ' = <b>' + S + '</b>';

      var direta = 0;
      for (var rr = y1; rr <= y2; rr++){
        for (var cc = x1; cc <= x2; cc++){
          direta += I[rr][cc];
        }
      }

      if (direta === S) {
        verifyEl.style.borderColor = '#a3e4d7';
        verifyEl.style.background  = '#eafaf1';
        verifyEl.style.color       = '#04342C';
      } else {
        verifyEl.style.borderColor = '#f5b7b1';
        verifyEl.style.background  = '#fdecea';
        verifyEl.style.color       = '#c0392b';
      }

      verifyEl.innerHTML = '&#10004; Verificação (Soma Direta dos Pixels) = ' + direta + (direta === S ? ' &rarr; Bate com S' : ' &rarr; Erro');
    }

    x1El.addEventListener('input', function(){ clampAndRender('x1'); });
    y1El.addEventListener('input', function(){ clampAndRender('y1'); });
    x2El.addEventListener('input', function(){ clampAndRender('x2'); });
    y2El.addEventListener('input', function(){ clampAndRender('y2'); });

    root.querySelectorAll('button[data-preset]').forEach(function(btn){
      btn.addEventListener('click', function(){
        var p = btn.getAttribute('data-preset').split(',').map(Number);
        x1El.value = p[0]; y1El.value = p[1]; x2El.value = p[2]; y2El.value = p[3];
        render();
      });
    });

    render();
  }

  function tryInitSim08Ep03(){
    var root = document.getElementById('sim-ep0803');
    if (root) initSim08Ep03(root); else setTimeout(tryInitSim08Ep03, 200);
  }
  tryInitSim08Ep03();
})();
</script>
""")

**Figura 8.3:** Simulatore EP08_03: Somma Rettangolare in O(1) — Molteplici Situazioni di Bordo


In [ ]:
%%writefile EP08_03.py
# Codice Python

In [ ]:
TestSuite("EP08_03.py").run()

### EP08_04 🟢 IoU e Soppressione Non-Massima (NMS)

La figura di questa sezione ha mostrato l’effetto della Soppressione Non-Massima su un insieme di riquadri prodotti da un rilevatore di tipo *sliding window*: più rilevazioni ridondanti per oggetto sono state ridotte a un singolo riquadro per oggetto. Ti è stato affidato il compito di reimplementare, byte per byte, le due funzioni che hanno prodotto quel risultato — `calcola_iou` e `soppressione_non_massima` — per confermare, con le tue mani, esattamente i numeri presentati nel capitolo.

#### 📋 Linee Guida di Implementazione

1. **Input:** Leggi l’intero $N$ (numero di riquadri) e il reale $\tau$ (soglia IoU). Successivamente, leggi $N$ righe, ciascuna con cinque reali $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.
2. **Intersezione su Unione:** Per due riquadri $A$ e $B$,
   $$
   \mathrm{IoU}(A,B) = \frac{\text{area}(A \cap B)}{\text{area}(A \cup B)},
   $$
   con area di intersezione nulla quando i riquadri non si sovrappongono.
3. **Algoritmo NMS** (esattamente come descritto nel capitolo):

   a. Ordina i riquadri per `score` decrescente (i pareggi mantengono l’ordine di lettura originale).

   b. Seleziona il riquadro con il punteggio più alto tra quelli rimanenti; aggiungilo all’output e rimuovilo dalla lista.

   c. Scarta, dalla lista rimanente, **tutti** i riquadri il cui IoU con il riquadro selezionato sia **maggiore o uguale** a $\tau$ — solo i riquadri con $\mathrm{IoU} < \tau$ rimangono candidati.

   d. Ripeti i passaggi (b)–(c) finché la lista dei rimanenti non è vuota.

4. **Output:** Per ogni riquadro mantenuto, nell’ordine in cui è stato selezionato, stampa il suo indice originale (posizione di lettura, a partire da $0$) e il suo `score`, con 2 cifre decimali. Alla fine, stampa `Totale mantenuti: X`.

#### 📌 Vincoli Computazionali

* **Attenzione alla direzione della soglia:** contrariamente a quanto si potrebbe supporre, un riquadro viene **soppresso** quando $\mathrm{IoU} \ge \tau$ (non solo quando $\mathrm{IoU} > \tau$) — segui esattamente questo criterio, lo stesso del codice di riferimento del capitolo.
* **Indici originali:** l’output fa riferimento alla posizione di lettura di ciascun riquadro nell’input, non alla sua posizione dopo l’ordinamento per `score`.
* **Area senza somma di 1 pixel:** usa area $= (x_{max}-x_{min}) \times (y_{max}-y_{min})$, esattamente come nel capitolo (senza l’aggiustamento “+1” talvolta usato in altre convenzioni).

#### 🧠 Fondamenti Teorici

| Elemento | Ruolo nel post-elaborazione |
|---|---|
| IoU | Quantifica la sovrapposizione spaziale tra due riquadri delimitatore |
| *Sliding window* (Haar Cascade) | Produce tipicamente più rilevazioni sovrapposte per lo stesso oggetto, in posizioni e scale vicine |
| Soglia $\tau$ | Controlla l’aggressività della soppressione: troppo bassa fonde oggetti vicini; troppo alta lascia passare ridondanze |
| Ordinamento per `score` | Garantisce che, tra riquadri ridondanti, quello con maggiore confidenza sopravviva sempre |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $N$ e reale $\tau$.
* Prossime $N$ righe: cinque reali $x_{min}\ y_{min}\ x_{max}\ y_{max}\ \text{score}$.

**Output:**

* Una riga per riquadro mantenuto, nell’ordine di selezione: `indice score` (score con 2 cifre decimali).
* Ultima riga: `Totale mantenuti: X`.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 5 0.4<br>50 50 150 150 0.90<br>60 55 155 145 0.75<br>58 60 160 150 0.60<br>300 300 400 420 0.95<br>310 305 395 415 0.70 | 3 0.95<br>0 0.90<br>Totale mantenuti: 2 | Esattamente l’esempio della figura del capitolo: 5 riquadri ridondanti (2 oggetti) diventano 2 rilevazioni finali. L’IoU tra il 1° e il 2° riquadro è $\approx 0{,}775$, ben al di sopra di $\tau=0{,}4$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0804" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0804 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0804 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0804 button:hover { background: #e8dfcf; }
  #sim-ep0804 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0804_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0804_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP08_04: IoU e Soppressione Non-Massima (NMS)</span>
  <span class="sim-ep0804_pill">Soppressione se IoU &ge; &tau;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0804_panel" style="margin-bottom:14px;">
    <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:12px;">
      
      <div>
        <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Spostamento della Candidata (dx)</label>
          <span id="sim-ep0804_dx_v" style="font-family:monospace; font-weight:700; color:#26241d;">3</span>
        </div>
        <input id="sim-ep0804_dx" type="range" min="0" max="10" step="1" value="3">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Soglia (&tau;)</label>
          <span id="sim-ep0804_tau_v" style="font-family:monospace; font-weight:700; color:#26241d;">0.40</span>
        </div>
        <input id="sim-ep0804_tau" type="range" min="0.1" max="0.9" step="0.05" value="0.4">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      La casella blu (punteggio maggiore) è già stata selezionata. Regola la sovrapposizione e la soglia &tau; per verificare la soppressione della casella rossa (candidata).
    </div>
  </div>

  <!-- Canvas Visual de Caixas Delimitadoras -->
  <div class="sim-ep0804_panel" style="position:relative; width:100%; height:160px; margin-bottom:14px; overflow:hidden;">
    <div id="sim-ep0804_boxA" style="position:absolute; border:2px solid #2980b9; background:rgba(41,128,185,0.20); border-radius:4px; transition:all 0.15s ease;"></div>
    <div id="sim-ep0804_boxB" style="position:absolute; border:2px solid #c0392b; background:rgba(192,57,43,0.20); border-radius:4px; transition:all 0.15s ease;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0804_debug" class="sim-ep0804_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep04(root){
    if (!root || root.dataset.sim08Ep04Init) return;
    root.dataset.sim08Ep04Init = "1";

    var dxEl   = root.querySelector('#sim-ep0804_dx');
    var dxvEl  = root.querySelector('#sim-ep0804_dx_v');
    var tauEl  = root.querySelector('#sim-ep0804_tau');
    var tauvEl = root.querySelector('#sim-ep0804_tau_v');
    var boxA   = root.querySelector('#sim-ep0804_boxA');
    var boxB   = root.querySelector('#sim-ep0804_boxB');
    var dbg    = root.querySelector('#sim-ep0804_debug');

    var ESCALA = 10;
    var A = {x1: 5, y1: 3, x2: 15, y2: 13};

    function iou(a, b){
      var ix1 = Math.max(a.x1, b.x1), iy1 = Math.max(a.y1, b.y1);
      var ix2 = Math.min(a.x2, b.x2), iy2 = Math.min(a.y2, b.y2);
      var iw  = Math.max(0, ix2 - ix1), ih = Math.max(0, iy2 - iy1);
      var inter = iw * ih;
      var areaA = (a.x2 - a.x1) * (a.y2 - a.y1);
      var areaB = (b.x2 - b.x1) * (b.y2 - b.y1);
      return inter / (areaA + areaB - inter);
    }

    function render(){
      var dx  = parseInt(dxEl.value, 10);
      var tau = parseFloat(tauEl.value);

      dxvEl.textContent  = dx;
      tauvEl.textContent = tau.toFixed(2);

      var B = {x1: 5 + dx, y1: 3 + dx * 0.4, x2: 15 + dx, y2: 13 + dx * 0.4};

      boxA.style.left   = (A.x1 * ESCALA) + 'px';
      boxA.style.top    = (A.y1 * ESCALA) + 'px';
      boxA.style.width  = ((A.x2 - A.x1) * ESCALA) + 'px';
      boxA.style.height = ((A.y2 - A.y1) * ESCALA) + 'px';

      boxB.style.left   = (B.x1 * ESCALA) + 'px';
      boxB.style.top    = (B.y1 * ESCALA) + 'px';
      boxB.style.width  = ((B.x2 - B.x1) * ESCALA) + 'px';
      boxB.style.height = ((B.y2 - B.y1) * ESCALA) + 'px';

      var val = iou(A, B);
      var suprimida = val >= tau;

      if (suprimida) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'IoU(A,B) = ' + val.toFixed(4) + '  |  \u03C4 = ' + tau.toFixed(2) + '  \u2192  Candidata (Vermelha) ' + 
        (suprimida ? 'SUPRIMIDA (IoU \u2265 \u03C4)' : 'MANTIDA (IoU < \u03C4)');
    }

    dxEl.addEventListener('input', render);
    tauEl.addEventListener('input', render);
    render();
  }

  function tryInitSim08Ep04(){
    var root = document.getElementById('sim-ep0804');
    if (root) initSim08Ep04(root); else setTimeout(tryInitSim08Ep04, 200);
  }
  tryInitSim08Ep04();
})();
</script>
""")

**Figura 8.4:** Simulatore EP08_04: IoU e Soppressione Non-Massimale


In [ ]:
%%writefile EP08_04.py
# Codice Python

In [ ]:
TestSuite("EP08_04.py").run()

### EP08_05 🟡 Etichettatura dei Componenti Connessi: Segmentazione delle Istanze

L'esempio di segmentazione classica di questo capitolo ha separato le "istanze" delle monete semplicemente tramite la loro disconnessione spaziale nella maschera binaria risultante dalla sogliatura di Otsu. Questa fase finale — etichettare ogni componente connesso con un identificatore di istanza — è esattamente ciò che ti è stato richiesto di implementare qui, da zero, su una maschera binaria già pronta (0 = sfondo, 1 = oggetto), come se fosse una reimplementazione manuale di `cv2.connectedComponents`.

Questo esercizio mette inoltre in luce, in modo molto concreto, la limitazione discussa nel capitolo: il risultato dipende interamente da come si definisce la "vicinanza" tra i pixel — e, come vedrai nel secondo esempio, due pixel in diagonale possono essere considerati la stessa istanza o istanze diverse, dipendendo esclusivamente dalla **connettività** scelta, non da alcuna nozione semantica di oggetto.

#### 📋 Linee Guida di Implementazione

1. **Input:** Leggere le dimensioni $H \times W$ della maschera binaria e i suoi $H \times W$ valori ($0$ o $1$).
2. **Connettività:** Leggere l'intero $c \in \{4, 8\}$. Con connettività $4$, i vicini di $(i,j)$ sono $(i{-}1,j)$, $(i{+}1,j)$, $(i,j{-}1)$ e $(i,j{+}1)$. Con connettività $8$, si aggiungono le quattro diagonali: $(i{-}1,j{-}1)$, $(i{-}1,j{+}1)$, $(i{+}1,j{-}1)$ e $(i{+}1,j{+}1)$.
3. **Scoperta dei componenti:** Scorrendo la maschera in scansione riga per riga, da sinistra a destra e dall'alto verso il basso, ogni volta che si incontra un pixel di valore $1$ non ancora etichettato, questo dà inizio a una **nuova componente**: assegnagli la prossima etichetta disponibile (la prima componente scoperta riceve l'etichetta $1$, la seconda l'etichetta $2$, e così via) e propaga la stessa etichetta a tutti i pixel di valore $1$ raggiungibili da esso tramite una catena di vicini (secondo la connettività scelta) — tramite ricerca in ampiezza, in profondità, o *union-find*, a tua scelta.
4. **Pixel di sfondo:** rimangono con etichetta $0$ e non appartengono a nessuna istanza.
5. **Output:** Prima, stampare la mappa completa delle etichette — $H$ righe con $W$ interi ciascuna. Successivamente, per ogni etichetta $\ell$ da $1$ a $K$ (nell'ordine di scoperta), stampare `Istanza l: A pixel`, dove $A$ è la quantità di pixel con quella etichetta. Infine, stampare `Totale istanze: K`.

#### 📌 Vincoli Computazionali

* **Ordine di scoperta = ordine di scansione:** le etichette sono numerate nell'ordine in cui ogni nuova componente viene trovata dalla scansione riga per riga, non per dimensione o posizione.
* **Connettività esplicita:** due pixel di valore $1$ appartengono alla stessa istanza solo se esiste una catena di vicini **secondo $c$** che li collega — non usare per errore la connettività opposta.
* **Maschera binaria pura:** tutti i valori di input sono esattamente $0$ o $1$.

#### 🧠 Fondamenti Teorici

| Elemento | Ruolo nella segmentazione classica delle istanze |
|---|---|
| Sogliatura (Otsu, Cap. 4) | Fase precedente che produce la maschera binaria a partire dall'immagine di intensità |
| Componente connessa | Ogni istanza è definita **solo** dalla connettività spaziale dei pixel dell'oggetto, senza alcuna nozione di forma, classe o aspetto |
| Connettività 4 vs. 8 | Parametro che altera il risultato: con connettività 8, due blob uniti solo in diagonale diventano un'unica istanza |
| Limitazione centrale | La tecnica fonde istanze che si toccano o si sovrappongono (anche se sono oggetti chiaramente distinti), poiché non c'è nozione di "oggetto" — solo di "regione connessa" |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $H$ e $W$.
* Prossime $H$ righe: $W$ interi ($0$ o $1$) ciascuna.
* Ultima riga: Intero $c$ ($4$ o $8$).

**Output:**

* $H$ righe con $W$ interi ciascuna (la mappa delle etichette).
* Una riga per istanza, nell'ordine di scoperta: `Istanza l: A pixel`.
* Ultima riga: `Totale istanze: K`.

#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 6 6<br>0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 1 1<br>0 0 0 0 1 1<br>8 | 0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 2 2<br>0 0 0 0 2 2<br>Istanza 1: 4 pixel<br>Istanza 2: 4 pixel<br>Totale istanze: 2 | Due blocchi $2\times2$ chiaramente separati: il risultato è lo stesso con connettività 4 o 8. |
| 2 2<br>1 0<br>0 1<br>8 | 1 0<br>0 1<br>Istanza 1: 2 pixel<br>Totale istanze: 1 | Con connettività 8, i due pixel in diagonale appartengono alla **stessa** istanza. Ripeti questo esempio con $c=4$: il risultato diventa 2 istanze di 1 pixel ciascuna — puramente per il cambiamento di connettività, senza alcuna differenza nella maschera. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0805" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0805 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0805 button { font-size: 11px; padding: 6px 16px; border-radius: 20px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0805 button:hover { background: #e8dfcf; }
  #sim-ep0805 button.sim-ep0805_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0805_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0805_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP08_05: Componenti Connessi (Connettività 4 vs. 8)</span>
  <span class="sim-ep0805_pill">Stessa Maschera &rarr; Etichette Diverse</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles de Seleção de Conectividade -->
  <div class="sim-ep0805_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      La stessa maschera (due pixel in diagonale) &mdash; alterna la connettività e osserva il numero di istanze e i colori delle etichette cambiare.
    </div>

    <div style="display:flex; gap:8px; justify-content:center;">
      <button id="sim-ep0805_c4">Connettività 4</button>
      <button id="sim-ep0805_c8" class="sim-ep0805_active">Connettività 8</button>
    </div>
  </div>

  <!-- Exibição da Grade 2x2 -->
  <div class="sim-ep0805_panel" style="margin-bottom:14px; display:flex; justify-content:center;">
    <div id="sim-ep0805_grid" style="display:grid; grid-template-columns:repeat(2, 56px); gap:6px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0805_debug" class="sim-ep0805_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim08Ep05(root){
    if (!root || root.dataset.sim08Ep05Init) return;
    root.dataset.sim08Ep05Init = "1";

    var mask = [[1, 0], [0, 1]];
    var conect = 8;
    var CORES = ['#eafaf1', '#fdecea'];
    var BORDAS = ['#a3e4d7', '#f5b7b1'];
    var TEXTOS = ['#04342C', '#c0392b'];

    var btn4   = root.querySelector('#sim-ep0805_c4');
    var btn8   = root.querySelector('#sim-ep0805_c8');
    var gridEl = root.querySelector('#sim-ep0805_grid');
    var dbg    = root.querySelector('#sim-ep0805_debug');

    function rotula(){
      var H = mask.length, W = mask[0].length;
      var labels = [[0, 0], [0, 0]];
      var atual = 0;
      var viz4 = [[-1, 0], [1, 0], [0, -1], [0, 1]];
      var viz8 = viz4.concat([[-1, -1], [-1, 1], [1, -1], [1, 1]]);
      var viz = conect === 8 ? viz8 : viz4;

      for (var i = 0; i < H; i++){
        for (var j = 0; j < W; j++){
          if (mask[i][j] === 1 && labels[i][j] === 0){
            atual++;
            var fila = [[i, j]];
            labels[i][j] = atual;
            while (fila.length){
              var pos = fila.pop();
              var r = pos[0], c = pos[1];
              for (var k = 0; k < viz.length; k++){
                var nr = r + viz[k][0], nc = c + viz[k][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W && mask[nr][nc] === 1 && labels[nr][nc] === 0){
                  labels[nr][nc] = atual;
                  fila.push([nr, nc]);
                }
              }
            }
          }
        }
      }
      return {labels: labels, k: atual};
    }

    function estiloBotoes(){
      btn4.classList.toggle('sim-ep0805_active', conect === 4);
      btn8.classList.toggle('sim-ep0805_active', conect === 8);
    }

    function render(){
      var res = rotula();
      gridEl.innerHTML = '';

      for (var i = 0; i < 2; i++){
        for (var j = 0; j < 2; j++){
          var d = document.createElement('div');
          var lab = res.labels[i][j];
          var estilo = 'width:56px; height:56px; display:flex; align-items:center; justify-content:center; border-radius:8px; font-family:monospace; font-weight:700; font-size:13px; transition:all 0.15s ease;';
          
          if (lab === 0){
            estilo += 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;';
          } else {
            var idx = (lab - 1) % 2;
            estilo += 'background:' + CORES[idx] + '; border:2px solid ' + BORDAS[idx] + '; color:' + TEXTOS[idx] + ';';
          }

          d.style.cssText = estilo;
          d.textContent = mask[i][j] + (lab ? ' (r' + lab + ')' : '');
          gridEl.appendChild(d);
        }
      }

      estiloBotoes();

      if (res.k === 1) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#26241d';
      }

      dbg.textContent = 'Connettività = ' + conect + '  \u2192  ' + res.k + ' Instância(s) Encontrada(s)';
    }

    btn4.addEventListener('click', function(){ conect = 4; render(); });
    btn8.addEventListener('click', function(){ conect = 8; render(); });

    render();
  }

  function tryInitSim08Ep05(){
    var root = document.getElementById('sim-ep0805');
    if (root) initSim08Ep05(root); else setTimeout(tryInitSim08Ep05, 200);
  }
  tryInitSim08Ep05();
})();
</script>
""")

**Figura 8.5:** Simulatore EP08_05: Etichettatura delle Componenti Connesse — Connettività 4 vs. 8


In [ ]:
%%writefile EP08_05.py
# Codice Python

In [ ]:
TestSuite("EP08_05.py").run()

### EP08_06 🟡 *Bounding Boxes*, Centroidi e Proprietà delle Istanze con `mm.measure`

Nell'esercizio precedente (**EP08_05**), si osserva come la segmentazione per componenti connesse etichetti regioni binarie contigue per separare le istanze. Tuttavia, per attività di rilevamento, tracciamento e analisi quantitativa degli oggetti, la semplice mappa delle etichette non è sufficiente. Diventa necessario estrarre **metriche spaziali e geometriche** che caratterizzino ciascuna istanza individualmente.

Questo EP si concentra sul calcolo e sull'estrazione automatica delle proprietà fondamentali della visione artificiale per ogni componente connessa trovata nella maschera binaria, utilizzando il metodo nativo `mm.measure(img)` della libreria `morph`:

1. **Bounding Box:** Il più piccolo rettangolo allineato agli assi che racchiude completamente l'istanza, definito dal suo angolo superiore sinistro $(x, y)$, larghezza $w$ e altezza $h$.
2. **Centroide Geometrico $(\bar{x}, \bar{y})$:** Il centro di massa dell'istanza sulla griglia discreta, equivalente ai momenti spaziali del primo ordine $M_{10}/M_{00}$ e $M_{01}/M_{00}$.
3. **Area Geometrica del Contorno ($A$):** L'area racchiusa dal contorno dell'istanza, calcolata tramite `mm.contourArea(c)`.



#### 📋 Linee Guida di Implementazione

1. **Input:** Leggere le dimensioni $H \times W$ della maschera binaria, i valori $H \times W$ ($0$ o $1$) e il parametro di connettività $c \in \{4, 8\}$.
2. **Estrazione Automatica con `mm.measure`:** Passare l'immagine binarizzata alla funzione `mm.measure(img_bin)`, che estrae i contorni OpenCV e restituisce una lista di dizionari contenenti le proprietà geometriche di ciascuna istanza.
3. **Proprietà Restituite:** Per ogni dizionario $m$ della lista restituita da `medidas = mm.measure(img_bin)`:
   * **Area (`area`):** Valore numerico dell'area geometrica del contorno `mm.contourArea(c)`.
   * **Bounding Box (`bbox`):** Tupla $(x, y, w, h)$ che rappresenta l'angolo superiore sinistro, la larghezza e l'altezza.
   * **Centroide (`center`):** Tupla $(c_x, c_y)$ con le coordinate del centro di massa $M_{10}/M_{00}$ e $M_{01}/M_{00}$. Formattare con **due cifre decimali**.
4. **Output:** Per ogni istanza $1, \dots, K$ trovata (ordinata per ordine di scoperta/posizione nell'immagine), stampare una riga contenente le sue proprietà. Infine, stampare il numero totale di istanze.
   - Per ordinare, usare `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`.



#### 🧠 Fondamenti Teorici

| Proprietà in `mm.measure` | Calcolo Matematico / Logica Discreta | Applicazione Pratica nella Visione Artificiale |
| --- | --- | --- |
| **`bbox` (OpenCV)** | $[x, y, w, h] = [\min(c), \min(r), \Delta c + 1, \Delta r + 1]$ | Formato classico di OpenCV. *Nota: reti come YOLO convertono questo rettangolo in $(c_x, c_y, w, h)$ normalizzato.* |
| **`center`** | $\bar{x} = \frac{M_{10}}{M_{00}}, \quad \bar{y} = \frac{M_{01}}{M_{00}}$ | Centro di massa esatto della maschera (usato nel tracciamento e nell'analisi delle traiettorie). |
| **`area`** | $A = \text{contourArea}(C)$ (Formula del Poligono) | Metrica continua della superficie dell'oggetto. |


#### 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $H$ e $W$.
* Prossime $H$ righe: $W$ interi ($0$ o $1$) ciascuna.
* Ultima riga: Intero $c$ ($4$ o $8$).

**Output:**

* Una riga per istanza nell'ordine di scoperta:
`Istanza l: Area=A, BBox=(x,y,w,h), Centroide=(cx,cy)`
* Ultima riga: `Totale istanze: K`.



#### 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 6 6<br>0 0 0 0 0 0<br>0 1 1 0 0 0<br>0 1 1 0 0 0<br>0 0 0 0 0 0<br>0 0 0 0 1 1<br>0 0 0 0 1 1<br>8 | Istanza 1: Area=1.0, BBox=(1,1,2,2), Centroide=(1.50,1.50)<br>Istanza 2: Area=1.0, BBox=(4,4,2,2), Centroide=(4.50,4.50)<br>Totale istanze: 2 | Blocchi $2\times2$ allineati. Il calcolo dell'area geometrica del contorno risulta in $1.0$. Il centroide del blocco nelle colonne 1–2 e righe 1–2 è esattamente $(1.50,\,1.50)$. |
| 4 6<br>0 0 0 0 0 0<br>0 1 1 1 1 0<br>0 0 0 1 0 0<br>0 0 0 0 0 0<br>4 | Istanza 1: Area=2.0, BBox=(1,1,4,2), Centroide=(2.40,1.20)<br>Totale istanze: 1 | Oggetto asimmetrico a forma di "T" invertita. L'area geometrica del contorno è $2.0$. Il centroide riflette la distribuzione dei pixel dell'oggetto. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0806" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulatore EP08_06: Metriche Morfologiche Native (mm.measure)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Contorno OpenCV & Momenti</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;justify-content:space-between;align-items:flex-end;margin-bottom:14px;flex-wrap:wrap;gap:10px;">
      <div>
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">AZIONE</div>
        <button id="ep0806_btnRandom" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Genera Nuove Istanze Binarie</button>
      </div>
      <div style="font-size:11px;color:#8a8371;font-family:monospace;">
        <span style="font-weight:700;color:#26241d;">Parametro di Precisione (approxPolyDP):</span> precisione = 0.01
      </div>
    </div>

    <!-- Container da Matriz de Píxeis -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">MAPPA DELLE ETICHETTE DELLE ISTANZE</div>
      <div id="ep0806_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <!-- Tabela de Métricas do mm.measure -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">METRICHE ESTRATTE DA MM.MEASURE</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimetro</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">centro (cx, cy)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circolarità</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidità</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertici</th>
            </tr>
          </thead>
          <tbody id="ep0806_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0806Init) return;
    root.dataset.ep0806Init = "1";

    var H = 10, W = 22;
    var mask = [], labels = [], metrics = [];
    var colors = ['#ffffff', '#7ee7c6', '#fca5a5', '#fde047', '#93c5fd', '#c084fc', '#f472b6'];

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [];
            var queue = [[r, c]];
            visited[r][c] = true;

            while (queue.length > 0) {
              var curr = queue.shift();
              var cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);

              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true;
                    queue.push([nr, nc]);
                  }
                }
              }
            }

            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) {
                borderPts.push([pc, pr]);
              }
            });

            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;

            borderPts.sort(function(a, b) {
              return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx);
            });

            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1];
        area -= contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length;
      var m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }

      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else {
          return [pts[0], pts[end]];
        }
      }
      return rdp(contour, epsilon);
    }

    function measureJS(mat) {
      var blobs = cv2_findContours(mat);
      var res = [];
      labels = Array.from({length: H}, function(){ return Array(W).fill(0); });

      blobs.forEach(function(item, idx) {
        var contour = item.contour;
        var pixels = item.pixels;
        var labelId = idx + 1;

        pixels.forEach(function(p){ labels[p[0]][p[1]] = labelId; });

        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;

        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);

        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;

        var poly = cv2_approxPolyDP(contour, 0.01);

        res.push({
          id: labelId,
          area: area,
          perimeter: per,
          cx: moments.cx,
          cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0,
          vertices: poly.length
        });
      });

      res.sort(function(a, b) {
        if (a.y !== b.y) return a.y - b.y;
        return a.x - b.x;
      });

      res.forEach(function(m, i) { m.id = i + 1; });
      return res;
    }

    function render() {
      var gridContainer = root.querySelector('#ep0806_grid_container');
      gridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var l = labels[r][c];
          var cell = document.createElement('div');
          var bg = l === 0 ? '#ffffff' : colors[(l % (colors.length - 1)) + 1];
          var fg = l === 0 ? '#8a8371' : '#26241d';
          cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;font-family:monospace;user-select:none;background:' + bg + ';color:' + fg + ';';
          cell.textContent = l;
          grid.appendChild(cell);
        }
      }
      gridContainer.appendChild(grid);

      var tbody = root.querySelector('#ep0806_tbody');
      tbody.innerHTML = '';

      if (metrics.length === 0) {
        tbody.innerHTML = '<tr><td colspan="8" style="padding:12px;color:#8a8371;text-align:center;">Nenhuma instância binária encontrada.</td></tr>';
        return;
      }

      metrics.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        var centerStr = '(' + m.cx.toFixed(2) + ', ' + m.cy.toFixed(2) + ')';
        var bboxStr = '(' + m.x + ', ' + m.y + ', ' + m.w + ', ' + m.h + ')';

        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + centerStr + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxStr + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        tbody.appendChild(tr);
      });
    }

    function generate() {
      mask = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var numObj = Math.floor(Math.random() * 2) + 2;

      for (var o = 0; o < numObj; o++) {
        var w = Math.floor(Math.random() * 3) + 3;
        var h = Math.floor(Math.random() * 3) + 3;
        var sr = Math.floor(Math.random() * (H - h));
        var sc = Math.floor(Math.random() * (W / numObj - w)) + Math.floor(o * (W / numObj));

        for (var r = 0; r < h; r++) {
          for (var c = 0; c < w; c++) {
            if (Math.random() > 0.15) mask[sr + r][sc + c] = 1;
          }
        }
      }

      metrics = measureJS(mask);
      render();
    }

    root.querySelector('#ep0806_btnRandom').addEventListener('click', generate);
    generate();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0806');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.6:** Simulatore EP08_06: Estrazione di *Bounding Boxes*, Centroidi e Proprietà con *mm.measure*


In [ ]:
%%writefile EP08_06.py
# Codice Python

In [ ]:
TestSuite("EP08_06.py").run()

### EP08_07 🟡 Rimozione del Rumore Sale e Pepe e Misurazione degli Oggetti

In questo esercizio, applicherai il filtraggio morfologico per pulire un'immagine binaria corrotta da rumore di tipo **sale e pepe** (pixel isolati di valore `1` nello sfondo e `0` all'interno degli oggetti). Dopo la pulizia, il programma deve estrarre le misurazioni geometriche dei componenti connessi rimanenti, ordinarli e visualizzare la tabella finale delle metriche.

#### 📋 Linee Guida di Implementazione

1. **Input:** leggere due interi $H$ e $W$ (altezza e larghezza dell'immagine) nella prima riga e, successivamente, le $H$ righe con la matrice binaria contenente pixel `0` e `1` separati da spazio.


2. **Filtraggio Morfologico:** applicare una sequenza di **Apertura** (per eliminare il rumore "sale" sullo sfondo) seguita da **Chiusura** (per riempire il rumore "pepe" all'interno degli oggetti) con elemento strutturante $3 \times 3$.
3. **Stampa dell'Immagine Pulita:** stampare la matrice risultante in valori `0` e `1` separati da spazio.


4. **Misurazioni Geometriche:** per ogni oggetto identificato nella matrice pulita, estrarre:
* `id`: identificatore numerico sequenziale (riassegnato dopo l'ordinamento);


* `area`: area calcolata tramite contorno (`cv2.contourArea`);


* `perimeter`: perimetro del contorno (`cv2.arcLength`);


* `cx`, `cy`: centro di massa (centroide tramite `cv2.moments`);


* `x`, `y`, `w`, `h`: coordinate del rettangolo delimitatore (`cv2.boundingRect`);


* `circularity`: circolarità data da $\frac{4 \pi \cdot \text{area}}{\text{perimetro}^2}$;
* `solidity`: solidità data dal rapporto $\frac{\text{area}}{\text{area dell'inviluppo convesso}}$;
* `vertices`: numero di vertici approssimato del poligono (`cv2.approxPolyDP` con $\epsilon = 0.02 \times \text{perimetro}$).


5. **Ordinamento e Uscita:** ordinare gli oggetti in ordine crescente in base alla posizione $X$ del rettangolo delimitatore (`bbox[0]`); in caso di parità, utilizzare la posizione $Y$ (`bbox[1]`). Riassegnare gli `id` da $1$ a $N$ e stampare la tabella formattata.
   - Per ordinare, utilizzare `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `medidas = mm.measure(img)`.



#### 📌 Vincoli e Regole di Ordinamento

* **Regola di Ordinamento degli Oggetti:**
```python
medidas.sort(key=lambda m: (m['bbox'][0], m['bbox'][1]))

```


* **Differenza di Area:** L'area calcolata da OpenCV (`cv2.contourArea`) misura l'area del poligono continuo delimitato dai centri dei pixel di bordo, risultando in valori numerici inferiori rispetto al semplice conteggio discreto dei pixel `1` (`np.sum`).


#### 🧠 Fondamenti Teorici

| Operazione / Metrica | Funzione nel Filtraggio e nella Caratterizzazione |
|--------------------|---------------------------------------|
| **Apertura Morfologica** ($\circ$) | Erosione seguita da dilatazione: rimuove rumori brillanti isolati (*sale*). |
| **Chiusura Morfologica** ($\bullet$) | Dilatazione seguita da erosione: riempie piccoli fori scuri all'interno degli oggetti (*pepe*). |
| **`cv2.boundingRect`** | Restituisce $(x, y, w, h)$, il più piccolo rettangolo allineato agli assi che racchiude l'oggetto. |
| **Circolarità e Solidità** | Descrivono la compattezza e la convessità geometrica del componente. |


#### 📌 Esempi

| Input | Output |
|---|---|
| 8 9<br>0 0 0 0 0 0 0 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 1 1<br>0 0 0 0 0 0 0 1 1<br>0 0 0 0 0 0 0 0 0 | id area perimeter cx cy x y w h circularity solidity vertices<br>1 9.0 12.0 3.5 2.0 3 1 4 3 0.79 1.000 4<br>2 4.0 8.0 7.5 5.5 7 5 2 2 0.79 1.000 4 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0807" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulatore EP08_07: Morfologia Commutabile (4-C / 8-C) & Metriche OpenCV</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Sale + Pepe &rarr; Apertura &rarr; Chiusura &rarr; Misurazione</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:260px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">FASE DELL'ELABORAZIONE MORFOLOGICA</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnOrig" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Con rumore</button>
          <button id="ep0807_btnAbert" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Apertura</button>
          <button id="ep0807_btnFech" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Chiusura</button>
        </div>
      </div>

      <div style="flex:1;min-width:140px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">ELEMENTO STRUTTURANTE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnConn4" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">4-Connesso</button>
          <button id="ep0807_btnConn8" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">8-Connesso</button>
        </div>
      </div>

      <div style="min-width:140px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">VISUALIZZAZIONE DEI PIXEL</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0807_btnVal" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">Valori (0/1)</button>
          <button id="ep0807_btnCor" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">Colori (P&W)</button>
        </div>
      </div>

      <div>
        <button id="ep0807_btnRandom" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Genera Scenario Casuale</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZZAZIONE DELLA MATRICE DEI PIXEL DI INGRESSO / ELABORATA</div>
      <div id="ep0807_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABELLA DELLE MISURAZIONI DEGLI OGGETTI (CALCOLATA DOPO APERTURA E CHIUSURA)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimetro</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circolarità</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidità</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertici</th>
            </tr>
          </thead>
          <tbody id="ep0807_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0807Init) return;
    root.dataset.ep0807Init = "1";

    var H = 12, W = 28;
    var imgBase = [], imgRuido = [], imgAbertura = [], imgLimpa = [];
    var medidasObjetos = [];
    var etapaAtual = 'ruido', modoExibicao = 'val', modoConectividade = 4;

    var elBtnOrig = root.querySelector('#ep0807_btnOrig');
    var elBtnAbert = root.querySelector('#ep0807_btnAbert');
    var elBtnFech = root.querySelector('#ep0807_btnFech');
    var elBtnConn4 = root.querySelector('#ep0807_btnConn4');
    var elBtnConn8 = root.querySelector('#ep0807_btnConn8');
    var elBtnVal = root.querySelector('#ep0807_btnVal');
    var elBtnCor = root.querySelector('#ep0807_btnCor');
    var elBtnRandom = root.querySelector('#ep0807_btnRandom');
    var elGridContainer = root.querySelector('#ep0807_grid_container');
    var elTbody = root.querySelector('#ep0807_tbody');

    var neighbors4 = [[0,0], [-1,0], [1,0], [0,-1], [0,1]];
    var neighbors8 = [[0,0], [-1,0], [1,0], [0,-1], [0,1], [-1,-1], [-1,1], [1,-1], [1,1]];

    function dilate(mat, conn) {
      var neighbors = conn === 8 ? neighbors8 : neighbors4;
      var res = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var hit = false;
          for (var i = 0; i < neighbors.length; i++) {
            var nr = r + neighbors[i][0], nc = c + neighbors[i][1];
            if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
              if (mat[nr][nc] === 1) { hit = true; break; }
            }
          }
          res[r][c] = hit ? 1 : 0;
        }
      }
      return res;
    }

    function erode(mat, conn) {
      var neighbors = conn === 8 ? neighbors8 : neighbors4;
      var res = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var fit = true;
          for (var i = 0; i < neighbors.length; i++) {
            var nr = r + neighbors[i][0], nc = c + neighbors[i][1];
            if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
              if (mat[nr][nc] !== 1) { fit = false; break; }
            } else { fit = false; }
          }
          res[r][c] = fit ? 1 : 0;
        }
      }
      return res;
    }

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.01);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function recalcularMorfologia() {
      imgAbertura = dilate(erode(imgRuido, modoConectividade), modoConectividade);
      imgLimpa = erode(dilate(imgAbertura, modoConectividade), modoConectividade);
      medidasObjetos = measureOpenCV(imgLimpa);
      renderGrid();
      renderTabela();
    }

    function gerarCenarioAleatorio() {
      imgBase = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var numObjetos = Math.floor(Math.random() * 2) + 2; 
      var setores = [ { minC: 1, maxC: 8 }, { minC: 10, maxC: 17 }, { minC: 19, maxC: 25 } ];
      var objetoPixels = [];
      for (var o = 0; o < numObjetos; o++) {
        var setor = setores[o];
        var tipoForma = Math.floor(Math.random() * 3);
        var objW = Math.floor(Math.random() * 2) + 4, objH = Math.floor(Math.random() * 2) + 4;
        var startC = Math.floor(Math.random() * (setor.maxC - setor.minC - objW + 1)) + setor.minC;
        var startR = Math.floor(Math.random() * (H - 4 - objH + 1)) + 2;
        for (var r = 0; r < objH; r++) {
          for (var c = 0; c < objW; c++) {
            var pr = startR + r, pc = startC + c, isObj = false;
            if (tipoForma === 0) isObj = true;
            else if (tipoForma === 1) { if (r >= objH / 2 || c < objW / 2) isObj = true; }
            else if (tipoForma === 2) { if (r < objH / 2 || (c >= Math.floor(objW / 3) && c <= Math.floor(2 * objW / 3))) isObj = true; }
            if (isObj) { imgBase[pr][pc] = 1; objetoPixels.push([pr, pc]); }
          }
        }
      }
      imgRuido = JSON.parse(JSON.stringify(imgBase));
      var qtdSal = Math.floor(Math.random() * 2) + 2;
      for (var s = 0; s < qtdSal; s++) {
        var sr = Math.floor(Math.random() * (H - 2)) + 1, sc = Math.floor(Math.random() * (W - 2)) + 1;
        if (imgBase[sr][sc] === 0) imgRuido[sr][sc] = 1;
      }
      var shuffledObj = objetoPixels.filter(function(p){ return p[0] > 0 && p[0] < H-1 && p[1] > 0 && p[1] < W-1; }).sort(function() { return 0.5 - Math.random(); });
      var qtdPimenta = Math.max(1, Math.floor(shuffledObj.length * 0.10));
      for (var p = 0; p < qtdPimenta; p++) { imgRuido[shuffledObj[p][0]][shuffledObj[p][1]] = 0; }
      recalcularMorfologia();
    }

    function renderGrid(){
      var mat = imgRuido;
      if (etapaAtual === 'abertura') mat = imgAbertura;
      if (etapaAtual === 'fechamento') mat = imgLimpa;
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var val = mat[r][c], cell = document.createElement('div');
          var bg = val === 1 ? '#26241d' : '#ffffff';
          var fg = val === 1 ? '#7ee7c6' : '#8a8371';
          cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;font-family:monospace;user-select:none;background:' + bg + ';color:' + fg + ';';
          if (modoExibicao === 'val') cell.textContent = val; else cell.textContent = '';
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o processamento morfológico.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setEtapa(etapa, btn){
      [elBtnOrig, elBtnAbert, elBtnFech].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      etapaAtual = etapa; renderGrid();
    }

    function setConectividade(conn, btn){
      [elBtnConn4, elBtnConn8].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      modoConectividade = conn; recalcularMorfologia();
    }

    function setModo(modo, btn){
      [elBtnVal, elBtnCor].forEach(function(b){ b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600'; });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      modoExibicao = modo; renderGrid();
    }

    elBtnOrig.addEventListener('click', function(){ setEtapa('ruido', elBtnOrig); });
    elBtnAbert.addEventListener('click', function(){ setEtapa('abertura', elBtnAbert); });
    elBtnFech.addEventListener('click', function(){ setEtapa('fechamento', elBtnFech); });
    elBtnConn4.addEventListener('click', function(){ setConectividade(4, elBtnConn4); });
    elBtnConn8.addEventListener('click', function(){ setConectividade(8, elBtnConn8); });
    elBtnVal.addEventListener('click', function(){ setModo('val', elBtnVal); });
    elBtnCor.addEventListener('click', function(){ setModo('cor', elBtnCor); });
    elBtnRandom.addEventListener('click', function(){ gerarCenarioAleatorio(); });

    gerarCenarioAleatorio();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0807');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.7:** Simulatore EP08_07: Morfologia con Connettività Configurabile e Misurazione


In [ ]:
%%writefile EP08_07.py
# Codice Python

In [ ]:
TestSuite("EP08_07.py").run()

### EP08_08 🟡 Immagine in Scala di Grigi e Soglia Dinamica

In questo esercizio, l'immagine di ingresso non è più strettamente binaria (`0`/`1`) ma diventa un'**immagine in scala di grigi ($8$ bit, $0\dots255$)**, dove gli oggetti hanno un'intensità media intermedia su uno sfondo scuro ($0$), oltre a rumore di tipo sale e pepe distribuito su tutta l'immagine.

#### 📋 Linee Guida di Implementazione

1. **Ingresso:** leggere $H$ e $W$ nella prima riga, seguiti dalle $H$ righe con valori interi da $0$ a $255$ in una matrice $H \times W$.
2. **Pre-elaborazione:**
* Applicare un filtro di **Mediana ($3 \times 3$)** per rimuovere il rumore sale e pepe mantenendo i bordi nitidi.
* Applicare la **Soglia di Otsu** (o una soglia fissa $T = 60$) per binarizzare l'immagine pulita.

3. **Misurazione e Uscita:** estrarre il contorno degli oggetti, calcolare le metriche geometriche (`area`, `perimeter`, `cx`, `cy`, `x`, `y`, `w`, `h`, `circularity`, `solidity`) e ordinare gli oggetti per `bbox[0]` (con `bbox[1]` come criterio di parità). Riassegnare `id` da $1$ a $N$ e stampare la tabella.
   - Per ordinare, usare `misure.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `misure = mm.measure(img)`.

#### 📌 Esempi

| Ingresso | Uscita |
|---|---|
| 16 32<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>(immagine binaria contenente un quadrato e un cerchio) | id area perimeter cx cy x y w h circularity solidity vertices<br>1 16.0 16.0 8.0 8.0 6 6 5 5 0.79 1.000 4<br>2 28.3 18.8 22.5 8.0 19 5 7 7 1.00 1.000 8 |

\newpage

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0808" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulatore EP08_08: Rumore Sale e Pepe in Scala di Grigi & Misurazione OpenCV</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Mediana 3x3 &rarr; Binarizzazione &rarr; Misurazione</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:260px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">FASE DI ELABORAZIONE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0808_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Grigio con Rumore</button>
          <button id="ep0808_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Mediana 3x3</button>
          <button id="ep0808_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Binarizzata (Otsu)</button>
        </div>
      </div>

      <div>
        <button id="ep0808_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Genera Forme/Posizioni Casuali</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZZAZIONE DELLA MATRICE DEI PIXEL</div>
      <div id="ep0808_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABELLA DELLE MISURAZIONI DEGLI OGGETTI (ORDINATI PER BBOX_X, BBOX_Y)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimetro</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circolarità</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidità</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0808_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0808Init) return;
    root.dataset.ep0808Init = "1";

    var H = 10, W = 20, stage = 0;
    var matOrig = [], matMed = [], matBin = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0808_btnRand');
    var elGridContainer = root.querySelector('#ep0808_grid_container');
    var elTbody = root.querySelector('#ep0808_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var w1 = Math.floor(Math.random() * 2) + 4, h1 = Math.floor(Math.random() * 2) + 4;
      var x1 = Math.floor(Math.random() * 2) + 1, y1 = Math.floor(Math.random() * 2) + 1;
      var val1 = 150;
      for (var r = y1; r < y1 + h1; r++) { for (var c = x1; c < x1 + w1; c++) matOrig[r][c] = val1; }

      var w2 = Math.floor(Math.random() * 2) + 4, h2 = Math.floor(Math.random() * 2) + 4;
      var x2 = Math.floor(Math.random() * 2) + 11, y2 = Math.floor(Math.random() * 2) + 2;
      var val2 = 180;
      for (var r = y2; r < y2 + h2; r++) { for (var c = x2; c < x2 + w2; c++) matOrig[r][c] = val2; }

      for (var i = 0; i < 4; i++) {
        var sr = Math.floor(Math.random() * H), sc = Math.floor(Math.random() * W);
        if (matOrig[sr][sc] === 0) matOrig[sr][sc] = 255;
      }
      matOrig[y1 + 1][x1 + 1] = 0; matOrig[y2 + 1][x2 + 1] = 0;

      matMed = JSON.parse(JSON.stringify(matOrig));
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var vals = [];
          for (var dr = -1; dr <= 1; dr++) {
            for (var dc = -1; dc <= 1; dc++) {
              var nr = r + dr, nc = c + dc;
              if (nr >= 0 && nr < H && nc >= 0 && nc < W) { vals.push(matOrig[nr][nc]); } else { vals.push(0); }
            }
          }
          vals.sort(function(a, b){ return a - b; });
          matMed[r][c] = vals[4];
        }
      }

      matBin = matMed.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      medidasObjetos = measureOpenCV(matBin);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matOrig : (stage === 1 ? matMed : matBin);

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          if (stage === 2) {
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else {
            var fgCinza = v > 128 ? '#000000' : '#ffffff';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + v + ',' + v + ',' + v + ');color:' + fgCinza + ';';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após a filtragem.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0808_stage0'), root.querySelector('#ep0808_stage1'), root.querySelector('#ep0808_stage2')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0808_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0808_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0808_stage2').addEventListener('click', function(){ setStage(2, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0808');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.8:** Simulatore EP08_08: Filtraggio Mediano in Scala di Grigi e Misurazione di Oggetti OpenCV


In [ ]:
%%writefile EP08_08.py
# Codice Python

In [ ]:
TestSuite("EP08_08.py").run()

### EP08_09 🟠 Gradiente di Illuminazione e Soglia Adattativa

In questa variazione, gli oggetti sono immersi in uno sfondo con **illuminazione non uniforme (gradiente morbido di illuminazione)**. La sogliatura semplice a valore singolo fallisce, richiedendo una pre-elaborazione più robusta.

#### 📋 Linee guida di implementazione

1. **Input:** immagine in scala di grigi $H \times W$ con variazione di sfondo da $20$ a $180$.
2. **Pre-elaborazione:**
  
* Applicare **Soglia Adattativa** (es.: `cv2.adaptiveThreshold` con finestra gaussiana di $15 \times 15$ e costante $C = 3$) per isolare gli oggetti indipendentemente dalla variazione dello sfondo.
  
  `cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, ksize, C) // 255`

  `ksize` e `C` vengono letti dopo l'immagine.
  
* Operazione morfologica di **Chiusura** ($3 \times 3$) per sigillare eventuali difetti nei contorni.


3. **Misurazione e Classificazione:** estrarre le misure.


4. **Ordinamento e Output:** ordinare per `(bbox[0], bbox[1])` e stampare la tabella includendo la colonna `class`.
   - Per ordinare, usare `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `medidas = mm.measure(img, precision=0.02)`.



#### 📌 Esempi

| Input | Output |
|---|---|
| 16 32<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>k 20 | id area perimeter cx cy x y w h circularity solidity vertices<br>1 9.0 12.0 10.0 5.0 8 3 5 5 0.79 1.000 4<br>2 28.3 18.8 25.0 12.0 22 9 7 7 1.00 1.000 3|

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0809" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulatore EP08_09: Gradiente di Illuminazione e Soglia Adattativa & Misurazione OpenCV</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Adattativo vs Globale &rarr; Misurazione</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:280px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">FASE DEL PROCESSAMENTO</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0809_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Gradiente in Grigio</button>
          <button id="ep0809_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Soglia Globale Fallita</button>
          <button id="ep0809_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Soglia Adattativa OK</button>
        </div>
      </div>

      <div>
        <button id="ep0809_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Genera Forme/Posizioni Casuali</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZZAZIONE DELLA MATRICE DI PIXEL</div>
      <div id="ep0809_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABELLA DELLE MISURAZIONI DEGLI OGGETTI (CALCOLATA CON SOGLIA ADATTATIVA OK)</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimetro</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circolarità</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidità</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertici</th>
            </tr>
          </thead>
          <tbody id="ep0809_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0809Init) return;
    root.dataset.ep0809Init = "1";

    var H = 10, W = 20, stage = 0;
    var matGrad = [], matGlob = [], matAdapt = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0809_btnRand');
    var elGridContainer = root.querySelector('#ep0809_grid_container');
    var elTbody = root.querySelector('#ep0809_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matGrad = Array.from({length: H}, function(_, r){
        return Array.from({length: W}, function(_, c){ return Math.round(20 + c * 9.5); });
      });
      var w1 = 3, h1 = 3;
      var x1 = Math.floor(Math.random() * 2) + 2, y1 = Math.floor(Math.random() * 2) + 2;
      for (var r = y1; r < y1 + h1; r++) { for (var c = x1; c < x1 + w1; c++) matGrad[r][c] += 90; }

      var w2 = 3, h2 = 3;
      var x2 = Math.floor(Math.random() * 2) + 14, y2 = Math.floor(Math.random() * 2) + 2;
      for (var r = y2; r < y2 + h2; r++) { for (var c = x2; c < x2 + w2; c++) matGrad[r][c] += 90; }

      matGlob = matGrad.map(function(row) { return row.map(function(v) { return v > 110 ? 1 : 0; }); });

      var blockSize = 5, half = Math.floor(blockSize / 2), C_val = 20;
      matAdapt = Array.from({length: H}, function(){ return Array(W).fill(0); });
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var sum = 0, count = 0;
          for (var dr = -half; dr <= half; dr++) {
            for (var dc = -half; dc <= half; dc++) {
              var nr = r + dr, nc = c + dc;
              if (nr >= 0 && nr < H && nc >= 0 && nc < W) { sum += matGrad[nr][nc]; count++; }
            }
          }
          var mean = sum / count;
          matAdapt[r][c] = matGrad[r][c] > (mean + C_val) ? 1 : 0;
        }
      }
      medidasObjetos = measureOpenCV(matAdapt);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matGrad : (stage === 1 ? matGlob : matAdapt);

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          if (stage > 0) {
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else {
            var fgCinza = v > 120 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o limiar adaptativo.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0809_stage0'), root.querySelector('#ep0809_stage1'), root.querySelector('#ep0809_stage2')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0809_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0809_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0809_stage2').addEventListener('click', function(){ setStage(2, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0809');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.9:** Simulatore EP08_09: Gradiente di Illuminazione, Soglia Adattativa e Misurazione OpenCV


In [ ]:
%%writefile EP08_09.py
# Codice Python

In [ ]:
TestSuite("EP08_09.py").run()

### EP08_10 🔴 Contrasto Basso e Separazione degli Oggetti Tangenti (*Watershed* / Distanza)

In questo esercizio, **alcuni oggetti geometrici sono leggermente a contatto (sovrapposti sui bordi)**. La semplice estrazione dei contorni tratterebbe due oggetti come uno solo.

#### 📋 Linee Guida di Implementazione

1. **Input:** matrice $H \times W$ in livelli di grigio con oggetti di intensità $110\dots140$ su sfondo $0$, con rumore e coppie di oggetti tangenti.
2. **Pre-elaborazione e Separazione:**
* Applicazione della sogliatura.
* Applicazione della **Trasformata della Distanza** (`mm.dist`).
* Ottenimento dei picchi di distanza per servire come marcatori nella **Trasformata *Watershed*** (`mm.watershed`), separando fisicamente gli oggetti a contatto nella maschera. **Suggerimento:** usare `mm.regmax()` per ottenere i massimi locali e successivamente etichettare con `mm.label0`.
* Dopo il *watershed*, applicare nuovamente la sogliatura con `mm.threshold(water,0)//255`.

3. **Analisi delle Componenti Connesse:** misurare ogni regione isolata post-*Watershed*.
4. **Output:** stampare i componenti ordinati per `(bbox[0], bbox[1])` con le loro metriche individuali di area, centroide e solidità.
   - Per ordinare, usare `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `medidas = mm.measure(img, precision=0.02)`.

#### 📌 Esempi

| Input | Output |
|---|---|
| 16 16<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>...<br>(immagine binaria contenente due quadrati) | id area perimeter cx cy x y w h solidity<br>1 16.0 16.0 5.0 5.0 3 3 5 5 1.000<br>2 16.0 16.0 11.0 5.0 9 3 5 5 1.000 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0810" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulatore EP08_10: Separazione di Dischi Tangenti (Trasformata L2 & Watershed)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">mm.dist L2 &rarr; mm.watershed &rarr; Misurazione</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:2;min-width:300px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">FASE DELL'ELABORAZIONE MORFOLOGICA</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0810_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Grigio Rumoroso</button>
          <button id="ep0810_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Maschera Unita</button>
          <button id="ep0810_stage2" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">3. Distanza L2</button>
          <button id="ep0810_stage3" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">4. Watershed (Taglio)</button>
        </div>
      </div>

      <div>
        <button id="ep0810_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Genera Dischi con Raggi Casuali</button>
      </div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZZAZIONE DELLA MATRICE DEI PIXEL</div>
      <div id="ep0810_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">TABELLA DELLE MISURAZIONI DEI DISCHI DOPO IL TAGLIO DEL WATERSHED</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">perimeter</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cx</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">cy</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">x</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">y</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">w</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">h</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">circularity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidity</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertices</th>
            </tr>
          </thead>
          <tbody id="ep0810_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0810Init) return;
    root.dataset.ep0810Init = "1";

    var H = 11, W = 21, stage = 0;
    var matOrig = [], matBin = [], matDist = [], matWash = [], medidasObjetos = [];

    var elBtnRand = root.querySelector('#ep0810_btnRand');
    var elGridContainer = root.querySelector('#ep0810_grid_container');
    var elTbody = root.querySelector('#ep0810_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_moments(blobPixels) {
      var m00 = blobPixels.length, m10 = 0.0, m01 = 0.0;
      blobPixels.forEach(function(p) { m10 += p[1]; m01 += p[0]; });
      return { m00: m00, cx: m00 > 0 ? m10 / m00 : 0, cy: m00 > 0 ? m01 / m00 : 0 };
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function measureOpenCV(mat) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var per = cv2_arcLength(contour);
        var bbox = cv2_boundingRect(pixels);
        var moments = cv2_moments(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        medidas.push({
          area: area, perimeter: per, cx: moments.cx, cy: moments.cy,
          x: bbox.x, y: bbox.y, w: bbox.w, h: bbox.h,
          circularity: per > 0 ? (4 * Math.PI * area) / (per * per) : 0,
          solidity: hull_area > 0 ? area / hull_area : 0, vertices: poly.length
        });
      });
      medidas.sort(function(a, b) { return a.x !== b.x ? a.x - b.x : a.y - b.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var r1 = Math.floor(Math.random() * 3) + 2, r2 = Math.floor(Math.random() * 3) + 2; 
      var cy1 = Math.floor(Math.random() * 2) + 4, cx1 = Math.floor(Math.random() * 2) + 3;
      var cx2 = cx1 + r1 + r2, cy2 = cy1;

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          var d1 = Math.hypot(r - cy1, c - cx1), d2 = Math.hypot(r - cy2, c - cx2);
          if (d1 <= r1 || d2 <= r2) { matOrig[r][c] = 140 + Math.floor(Math.random() * 20); }
        }
      }

      matBin = matOrig.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      matDist = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var fundoPixels = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) { if (matBin[r][c] === 0) fundoPixels.push([r, c]); }
      }

      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (matBin[r][c] === 1) {
            var minDist = Infinity;
            for (var k = 0; k < fundoPixels.length; k++) {
              var dist = Math.hypot(r - fundoPixels[k][0], c - fundoPixels[k][1]);
              if (dist < minDist) minDist = dist;
            }
            matDist[r][c] = Math.round(minDist);
          }
        }
      }

      matWash = JSON.parse(JSON.stringify(matBin));
      var colCorte = cx1 + r1; 
      for (var r = 0; r < H; r++) { if (matWash[r][colCorte] === 1) matWash[r][colCorte] = 0; }

      medidasObjetos = measureOpenCV(matWash);
      renderGrid();
      renderTabela();
    }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var cell = document.createElement('div');
          if (stage === 0) {
            var v = matOrig[r][c]; cell.textContent = v;
            var fgCinza = v > 120 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';';
          } else if (stage === 1) {
            var v = matBin[r][c]; cell.textContent = v;
            var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';';
          } else if (stage === 2) {
            var v = matDist[r][c]; cell.textContent = v;
            var bgDist = v > 0 ? 'rgb(' + (240 - v * 45) + ',' + (240 - v * 30) + ',255)' : '#ffffff';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgDist + ';color:#26241d;';
          } else {
            var v = matWash[r][c]; cell.textContent = v;
            var bgWash = v === 1 ? '#26241d' : '#ffffff', fgWash = v === 1 ? '#7ee7c6' : '#8a8371';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgWash + ';color:' + fgWash + ';';
          }
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="12" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado após o corte do Watershed.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.perimeter.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cx.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.cy.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.x + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.y + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.w + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.h + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.circularity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>';
        elTbody.appendChild(tr);
      });
    }

    function setStage(s, btn){
      [root.querySelector('#ep0810_stage0'), root.querySelector('#ep0810_stage1'), root.querySelector('#ep0810_stage2'), root.querySelector('#ep0810_stage3')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    elBtnRand.addEventListener('click', gerarCenario);
    root.querySelector('#ep0810_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0810_stage1').addEventListener('click', function(){ setStage(1, this); });
    root.querySelector('#ep0810_stage2').addEventListener('click', function(){ setStage(2, this); });
    root.querySelector('#ep0810_stage3').addEventListener('click', function(){ setStage(3, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0810');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.10:** Simulatore EP08_10: Separazione di Dischi Tangenti via Trasformata della Distanza L2 e *Watershed*


\newpage

In [ ]:
%%writefile EP08_10.py
# Codice Python

In [ ]:
TestSuite("EP08_10.py").run()

### EP08_11 🔴 Classificazione e Validazione di Oggetti con Riferimento di *Bounding Box*

In questo esercizio, l'obiettivo è elaborare un'immagine in scala di grigi contenente molteplici oggetti geometrici, estrarne le proprietà con `mm.measure` e validare le scatole delimitatrici (*bounding boxes*) rilevate rispetto a un riferimento reale (*Ground Truth* - GT) fornito in input, utilizzando la metrica IoU (*Intersection over Union*).

#### 📋 Linee Guida di Implementazione

1. **Lettura dell'Immagine:** Leggere le dimensioni $H \times W$ e la matrice $H \times W$ di pixel dell'immagine in scala di grigi.
2. ***Pipeline* Morfologico:** Binarizzare l'immagine tramite il metodo di Otsu (`mm.threshold`) e visualizzare la maschera binarizzata risultante utilizzando `mm.drawImg`.
3. **Lettura del Riferimento Reale (*Ground Truth*):**
   
* Leggere la quantità $G$ di scatole delimitatrici del riferimento.
* Se $G > 0$, leggere $G$ righe contenenti 5 valori ciascuna: `id xmin_norm ymin_norm xmax_norm ymax_norm`.
* **Conversione delle Coordinate:** Le coordinate del riferimento sono normalizzate nell'intervallo $[0.0, 1.0]$. Per convertirle in pixel sulla griglia dell'immagine:

$$x_{\min} = \lfloor \text{xmin\_norm} \times W \rfloor, \quad y_{\min} = \lfloor \text{ymin\_norm} \times H \rfloor$$

$$w = \lfloor \text{xmax\_norm} \times W \rfloor - x_{\min}, \quad h = \lfloor \text{ymax\_norm} \times H \rfloor - y_{\min}$$

4. **Estrazione delle Metriche e Calcolo dell'IoU:**
* Estrarre le proprietà delle istanze con `mm.measure(img_bin, precision=0.02)`.
* Per ogni *bounding box* rilevata $(x, y, w, h)$, calcolare la sovrapposizione IoU rispetto alle scatole del riferimento e definire `hits = 1` se esiste una corrispondenza con $\text{IoU} \ge 0.50$, oppure `hits = 0` in caso contrario.

5. **Output:** Ordinare le istanze per posizione `(bbox[0], bbox[1])` e stampare la tabella CSV con la colonna aggiuntiva `hits`.
   - Per ordinare, usare `medidas.sort(key=lambda m: (m['bbox'][1], m['bbox'][0]))`, con `medidas = mm.measure(img)`.

#### 🧠 Fondamenti Teorici e Conversione

| Concetto | Formula / Operazione | Descrizione |
| --- | --- | --- |
| **BBox Rilevata** | $(x, y, w, h)$ tramite `mm.measure` | Scatola delimitatrice calcolata sulla griglia discreta in pixel interi. |
| **BBox Riferimento (GT)** | $(x_{\min}, y_{\min}, w, h)$ convertiti | Scatola reale fornita in input in coordinate relative $[0.0, 1.0]$. |
| **IoU (Intersection over Union)** | $\text{IoU} = \frac{\text{Area}(B_{\text{DET}} \cap B_{\text{GT}})}{\text{Area}(B_{\text{DET}} \cup B_{\text{GT}})}$ | Valuta il tasso di sovrapposizione delle scatole. È considerata valida se $\text{IoU} \ge 0.50$. |
| **Stato di Validazione (`hits`)** | $1$ se $\max(\text{IoU}) \ge 0.50$, altrimenti $0$ | Indicatore binario di correttezza del rilevatore rispetto al riferimento. |

#### 📦 Specifica di Input e Output (VPL)

**Input:**

* **Riga 1:** Interi $H$ e $W$ (dimensioni della matrice).
* **Successive $H$ righe:** $W$ interi ($0$ a $255$) che rappresentano l'immagine in scala di grigi.
* **Riga $H + 2$:** Intero $G$ (quantità di scatole del riferimento reale).
* **Successive $G$ righe:** 5 valori numerici per riga: `id xmin_norm ymin_norm xmax_norm ymax_norm` (dove le coordinate sono valori fluttuanti tra $0.0$ e $1.0$).

**Output:**

1. Matrice binarizzata visualizzata tramite `mm.drawImg(img_bin)`.
2. Intestazione CSV: `id,area,perimeter,cx,cy,x,y,w,h,circularity,solidity,vertices,hits`
3. Una riga CSV per oggetto rilevato contenente le sue proprietà formattate e l'indicatore `hits` ($1$ o $0$).

#### 📌 Esempi

| Input | Output |
|---|---|
| 10 20<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 180 0 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 180 180 180 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 0 180 0 0 0 0 0 0 0 0 180 180 180 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>2<br>1 0.10 0.30 0.25 0.60<br>2 0.60 0.30 0.75 0.60 | 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 1 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 0 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0<br>id area perimeter cx cy x y w h circularity solidity vertices hits<br>1 2.0 5.7 3.0 4.0 2 3 3 3 0.79 1.000 4 1<br>2 4.0 8.0 13.0 4.0 12 3 3 3 0.79 1.000 4 1 |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0811" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulatore EP08_11: Bounding Boxes e Confronto IoU con Controlli Indipendenti</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">Validazione BBox GT vs DET</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:flex-end;margin-bottom:14px;">
      <div style="flex:1.5;min-width:220px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MODALITÀ DI VISUALIZZAZIONE</div>
        <div style="display:flex;gap:4px;background:#f1ead7;border:1px solid #e4dcc8;border-radius:11px;padding:3px;">
          <button id="ep0811_stage0" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:700;border:none;background:#26241d;color:#fbf7ee;cursor:pointer;border-radius:9px;white-space:nowrap;">1. Grigio Originale</button>
          <button id="ep0811_stage1" style="flex:1;text-align:center;padding:6px 10px;font-size:11px;font-weight:600;border:none;background:transparent;color:#8a8371;cursor:pointer;border-radius:9px;white-space:nowrap;">2. Binarizzata + Overlay</button>
        </div>
      </div>

      <div style="flex:1.5;min-width:240px;">
        <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">VISUALIZZAZIONE DELLE BOUNDING BOXES</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0811_toggleGT" style="padding:5px 10px;font-size:10.5px;font-weight:700;border:1px solid #1d4ed8;background:#2563eb;color:#ffffff;cursor:pointer;border-radius:8px;white-space:nowrap;display:inline-flex;align-items:center;gap:5px;"><span>🟦</span>BBox di Riferimento (GT)</button>
          <button id="ep0811_toggleDET" style="padding:5px 10px;font-size:10.5px;font-weight:700;border:1px solid #047857;background:#059669;color:#ffffff;cursor:pointer;border-radius:8px;white-space:nowrap;display:inline-flex;align-items:center;gap:5px;"><span>🟩</span>BBox Rilevata (DET)</button>
        </div>
      </div>

      <div>
        <button id="ep0811_btnRand" style="background:#26241d;color:#7ee7c6;border:none;padding:8px 14px;font-size:11px;font-weight:700;border-radius:9px;cursor:pointer;display:inline-flex;align-items:center;gap:6px;font-family:monospace;">🎲 Genera Scene Casuali</button>
      </div>
    </div>

    <div style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:10px;padding:8px 12px;margin-bottom:12px;display:flex;gap:16px;flex-wrap:wrap;align-items:center;justify-content:center;">
      <span style="font-size:9.5px;font-weight:700;color:#8a8371;margin-right:4px;">LEGENDA DELLE BBOXES:</span>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#2563eb;border:1px dashed #93c5fd;"></span> <span>Riferimento Reale (GT)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#059669;border:1px solid #34d399;"></span> <span>Rilevazione Accettata (IoU &ge; 0.5)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#dc2626;border:1px solid #f87171;"></span> <span>Rilevazione Rifiutata (IoU &lt; 0.5)</span></div>
      <div style="display:inline-flex;align-items:center;gap:5px;font-size:10.5px;font-weight:600;color:#374151;"><span style="width:12px;height:12px;border-radius:3px;display:inline-block;background:#7c3aed;border:1px double #a78bfa;"></span> <span>Sovrapposizione di BBoxes</span></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;margin-bottom:14px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:8px;text-align:left;">VISUALIZZAZIONE DELLA MATRICE DI PIXEL</div>
      <div id="ep0811_grid_container" style="overflow-x:auto;padding-bottom:4px;"></div>
    </div>

    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="font-size:9.5px;font-weight:700;color:#8a8371;margin-bottom:3px;letter-spacing:.3px;">MISURE, CLASSIFICAZIONE GEOMETRICA E CONFRONTO IoU CON RIFERIMENTO</div>
      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:10.5px;font-family:monospace;margin-top:10px;">
          <thead>
            <tr style="background:#f1ead7;">
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">id</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">classe</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">area</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">solidità</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">vertici</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox det (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">bbox gt (x,y,w,h)</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">IoU</th>
              <th style="padding:6px 4px;border:1px solid #e4dcc8;color:#26241d;font-weight:700;text-align:center;">stato (IoU &ge; 0.5)</th>
            </tr>
          </thead>
          <tbody id="ep0811_tbody"></tbody>
        </table>
      </div>
    </div>
  </div>
</div>

<script>
(function(){
  function initSim(root){
    if (!root || root.dataset.ep0811Init) return;
    root.dataset.ep0811Init = "1";

    var H = 10, W = 20, stage = 0;
    var showGT = true, showDET = true;
    // matOrig/matBin: cena. medidasObjetos: 1 registro por objeto real (GT), casado com sua melhor DET.
    // detBoxesAtuais: caixas "detectadas" simuladas (com ruído/deslocamento em relação ao objeto real).
    var matOrig = [], matBin = [], medidasObjetos = [], detBoxesAtuais = [];

    var elBtnRand = root.querySelector('#ep0811_btnRand');
    var elToggleGT = root.querySelector('#ep0811_toggleGT');
    var elToggleDET = root.querySelector('#ep0811_toggleDET');
    var elGridContainer = root.querySelector('#ep0811_grid_container');
    var elTbody = root.querySelector('#ep0811_tbody');

    function cv2_findContours(mat) {
      var visited = Array.from({length: H}, function(){ return Array(W).fill(false); });
      var contours = [];
      for (var r = 0; r < H; r++) {
        for (var c = 0; c < W; c++) {
          if (mat[r][c] === 1 && !visited[r][c]) {
            var blobPixels = [], queue = [[r, c]];
            visited[r][c] = true;
            while (queue.length > 0) {
              var curr = queue.shift(), cr = curr[0], cc = curr[1];
              blobPixels.push([cr, cc]);
              var dirs = [[-1,0],[1,0],[0,-1],[0,1],[-1,-1],[-1,1],[1,-1],[1,1]];
              for (var d = 0; d < dirs.length; d++) {
                var nr = cr + dirs[d][0], nc = cc + dirs[d][1];
                if (nr >= 0 && nr < H && nc >= 0 && nc < W) {
                  if (mat[nr][nc] === 1 && !visited[nr][nc]) {
                    visited[nr][nc] = true; queue.push([nr, nc]);
                  }
                }
              }
            }
            var borderPts = [];
            blobPixels.forEach(function(p) {
              var pr = p[0], pc = p[1];
              if (pr === 0 || pr === H-1 || pc === 0 || pc === W-1 ||
                  mat[pr-1][pc] === 0 || mat[pr+1][pc] === 0 ||
                  mat[pr][pc-1] === 0 || mat[pr][pc+1] === 0) { borderPts.push([pc, pr]); }
            });
            var cx = 0, cy = 0;
            borderPts.forEach(function(pt){ cx += pt[0]; cy += pt[1]; });
            cx /= borderPts.length; cy /= borderPts.length;
            borderPts.sort(function(a, b) { return Math.atan2(a[1] - cy, a[0] - cx) - Math.atan2(b[1] - cy, b[0] - cx); });
            contours.push({ pixels: blobPixels, contour: borderPts });
          }
        }
      }
      return contours;
    }

    function cv2_contourArea(contour) {
      if (contour.length < 3) return 0.0;
      var area = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        area += contour[i][0] * contour[j][1] - contour[j][0] * contour[i][1];
      }
      return Math.abs(area) / 2.0;
    }

    function cv2_arcLength(contour) {
      if (contour.length < 2) return 0.0;
      var per = 0.0;
      for (var i = 0; i < contour.length; i++) {
        var j = (i + 1) % contour.length;
        per += Math.hypot(contour[j][0] - contour[i][0], contour[j][1] - contour[i][1]);
      }
      return per;
    }

    function cv2_boundingRect(blobPixels) {
      var minX = W, maxX = 0, minY = H, maxY = 0;
      blobPixels.forEach(function(p) {
        var r = p[0], c = p[1];
        if (c < minX) minX = c; if (c > maxX) maxX = c;
        if (r < minY) minY = r; if (r > maxY) maxY = r;
      });
      return { x: minX, y: minY, w: maxX - minX + 1, h: maxY - minY + 1 };
    }

    function cv2_convexHull(points) {
      if (points.length <= 2) return points;
      var pts = points.slice().sort(function(a,b){ return a[0] === b[0] ? a[1] - b[1] : a[0] - b[0]; });
      function cross(o, a, b) { return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0]); }
      var lower = [];
      for (var i = 0; i < pts.length; i++) {
        while (lower.length >= 2 && cross(lower[lower.length - 2], lower[lower.length - 1], pts[i]) <= 0) lower.pop();
        lower.push(pts[i]);
      }
      var upper = [];
      for (var i = pts.length - 1; i >= 0; i--) {
        while (upper.length >= 2 && cross(upper[upper.length - 2], upper[upper.length - 1], pts[i]) <= 0) upper.pop();
        upper.push(pts[i]);
      }
      upper.pop(); lower.pop();
      return lower.concat(upper);
    }

    function cv2_approxPolyDP(contour, precision) {
      if (contour.length <= 2) return contour;
      var epsilon = precision * cv2_arcLength(contour);
      function rdp(pts, eps) {
        if (pts.length <= 2) return pts;
        var dmax = 0, index = 0, end = pts.length - 1;
        for (var i = 1; i < end; i++) {
          var dx = pts[end][0] - pts[0][0], dy = pts[end][1] - pts[0][1];
          var mag = Math.hypot(dx, dy);
          var d = mag === 0 ? Math.hypot(pts[i][0] - pts[0][0], pts[i][1] - pts[0][1]) :
            Math.abs(dy * pts[i][0] - dx * pts[i][1] + pts[end][0] * pts[0][1] - pts[end][1] * pts[0][0]) / mag;
          if (d > dmax) { index = i; dmax = d; }
        }
        if (dmax > eps) {
          var r1 = rdp(pts.slice(0, index + 1), eps);
          var r2 = rdp(pts.slice(index, end + 1), eps);
          return r1.slice(0, r1.length - 1).concat(r2);
        } else { return [pts[0], pts[end]]; }
      }
      return rdp(contour, epsilon);
    }

    function calculateIoU(boxA, boxB) {
      var ax1 = boxA.x, ay1 = boxA.y, ax2 = boxA.x + boxA.w, ay2 = boxA.y + boxA.h;
      var bx1 = boxB.x, by1 = boxB.y, bx2 = boxB.x + boxB.w, by2 = boxB.y + boxB.h;
      var ix1 = Math.max(ax1, bx1), iy1 = Math.max(ay1, by1);
      var ix2 = Math.min(ax2, bx2), iy2 = Math.min(ay2, by2);
      var iw = Math.max(0, ix2 - ix1), ih = Math.max(0, iy2 - iy1);
      var inter = iw * ih;
      var areaA = boxA.w * boxA.h, areaB = boxB.w * boxB.h;
      var union = areaA + areaB - inter;
      return union > 0 ? inter / union : 0.0;
    }

    // FIX: bbox extraída dos pixels reais do objeto (mat) é o GABARITO (GT) — é a posição
    // verdadeira e exata do objeto na cena. As caixas em `detBoxes` (com deslocamento
    // aleatório) representam a saída ruidosa de um detector, e são casadas ao GT mais
    // próximo por IoU.
    function measureOpenCV(mat, detBoxes) {
      var blobs = cv2_findContours(mat);
      var medidas = [];
      blobs.forEach(function(item) {
        var contour = item.contour, pixels = item.pixels;
        var area = cv2_contourArea(contour);
        if (area === 0) area = pixels.length;
        var gtBbox = cv2_boundingRect(pixels);
        var hull = cv2_convexHull(contour);
        var hull_area = cv2_contourArea(hull);
        if (hull_area === 0) hull_area = area;
        var poly = cv2_approxPolyDP(contour, 0.02);
        var solidity = hull_area > 0 ? area / hull_area : 0;
        var classe = solidity < 0.85 ? "Cruz (Côncavo)" : "Retângulo (Convexo)";
        var bestIoU = 0.0, matchedDET = { x: 0, y: 0, w: 0, h: 0 };

        detBoxes.forEach(function(d) {
          var iou = calculateIoU(gtBbox, d);
          if (iou > bestIoU) { bestIoU = iou; matchedDET = d; }
        });

        medidas.push({
          classe: classe, area: area, gtBox: gtBbox, detBox: matchedDET,
          solidity: solidity, vertices: poly.length, iou: bestIoU, ok: bestIoU >= 0.50
        });
      });
      medidas.sort(function(a, b) { return a.gtBox.x !== b.gtBox.x ? a.gtBox.x - b.gtBox.x : a.gtBox.y - b.gtBox.y; });
      medidas.forEach(function(m, i) { m.id = i + 1; });
      return medidas;
    }

    function gerarCenario() {
      matOrig = Array.from({length: H}, function(){ return Array(W).fill(0); });
      var armLen = Math.floor(Math.random() * 2) + 1, thick = 1;
      var w1 = armLen * 2 + thick, h1 = armLen * 2 + thick;
      var x1 = Math.floor(Math.random() * Math.max(1, 8 - w1)) + 1;
      var y1 = Math.floor(Math.random() * Math.max(1, H - h1)) + 1;

      for (var r = 0; r < h1; r++) {
        for (var c = 0; c < w1; c++) {
          if ((c >= armLen && c < armLen + thick) || (r >= armLen && r < armLen + thick)) { matOrig[y1 + r][x1 + c] = 180; }
        }
      }

      var w2 = Math.floor(Math.random() * 3) + 3, h2 = Math.floor(Math.random() * 3) + 3;
      var x2 = Math.floor(Math.random() * Math.max(1, W - 10 - w2)) + 10;
      var y2 = Math.floor(Math.random() * Math.max(1, H - h2)) + 1;

      for (var r = y2; r < y2 + h2; r++) {
        for (var c = x2; c < x2 + w2; c++) { matOrig[r][c] = 180; }
      }

      // Estas caixas simulam a saída de um DETECTOR real: deslocadas/imprecisas em relação
      // ao objeto verdadeiro (que será obtido depois via segmentação em matBin -> GT).
      var shiftX1 = Math.random() > 0.5 ? 1 : 0, shiftY1 = Math.random() > 0.5 ? 1 : 0;
      var shiftX2 = Math.random() > 0.6 ? -2 : 0;

      detBoxesAtuais = [
        { x: Math.max(0, x1 + shiftX1), y: Math.max(0, y1 + shiftY1), w: w1, h: h1 },
        { x: Math.max(0, x2 + shiftX2), y: y2, w: w2 + (shiftX2 !== 0 ? 2 : 0), h: h2 }
      ];

      matBin = matOrig.map(function(row) { return row.map(function(v) { return v > 50 ? 1 : 0; }); });
      medidasObjetos = measureOpenCV(matBin, detBoxesAtuais);
      renderGrid();
      renderTabela();
    }

    function inBox(r, c, box) { return r >= box.y && r < box.y + box.h && c >= box.x && c < box.x + box.w; }
    function isBoxEdge(r, c, box) { if (!inBox(r, c, box)) return false; return r === box.y || r === box.y + box.h - 1 || c === box.x || c === box.x + box.w - 1; }

    function renderGrid(){
      elGridContainer.innerHTML = '';
      var grid = document.createElement('div');
      grid.style.cssText = 'display:inline-grid;gap:1px;background:#e4dcc8;padding:1px;border-radius:6px;overflow:auto;max-width:100%;grid-template-columns:repeat(' + W + ', 22px);';
      var currentMat = stage === 0 ? matOrig : matBin;

      for (var r = 0; r < H; r++){
        for (var c = 0; c < W; c++){
          var v = currentMat[r][c], cell = document.createElement('div');
          var isGT = false, isDetOK = false, isDetFail = false;

          if (stage === 1) {
            // FIX: GT agora vem de m.gtBox (bbox real extraída dos pixels do objeto)
            if (showGT) { medidasObjetos.forEach(function(m) { if (isBoxEdge(r, c, m.gtBox)) isGT = true; }); }
            // FIX: DET agora vem de m.detBox (bbox ruidosa casada por IoU)
            if (showDET) {
              medidasObjetos.forEach(function(m) {
                if (isBoxEdge(r, c, m.detBox)) { if (m.ok) isDetOK = true; else isDetFail = true; }
              });
            }

            if (isGT && isDetOK) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#7c3aed;color:#ffffff;border:2px double #a78bfa;box-sizing:border-box;';
            } else if (isGT && isDetFail) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#c026d3;color:#ffffff;border:2px double #f472b6;box-sizing:border-box;';
            } else if (isDetOK) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#059669;color:#ffffff;border:2px solid #34d399;box-sizing:border-box;';
            } else if (isDetFail) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#dc2626;color:#ffffff;border:2px solid #f87171;box-sizing:border-box;';
            } else if (isGT) {
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:#2563eb;color:#ffffff;border:2px dashed #93c5fd;box-sizing:border-box;';
            } else {
              var bgBin = v === 1 ? '#26241d' : '#ffffff', fgBin = v === 1 ? '#7ee7c6' : '#8a8371';
              cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:' + bgBin + ';color:' + fgBin + ';border:none;';
            }
          } else {
            var fgCinza = v > 100 ? '#ffffff' : '#374151';
            cell.style.cssText = 'width:22px;height:22px;display:flex;align-items:center;justify-content:center;font-size:8px;font-weight:700;font-family:monospace;user-select:none;background:rgb(' + (255 - v) + ',' + (255 - v) + ',' + (255 - v) + ');color:' + fgCinza + ';border:none;';
          }
          cell.textContent = v;
          grid.appendChild(cell);
        }
      }
      elGridContainer.appendChild(grid);
    }

    function renderTabela(){
      elTbody.innerHTML = '';
      if (medidasObjetos.length === 0) {
        elTbody.innerHTML = '<tr><td colspan="9" style="padding:12px;color:#8a8371;text-align:center;">Nenhum objeto detectado na cena.</td></tr>';
        return;
      }
      medidasObjetos.forEach(function(m, i){
        var tr = document.createElement('tr');
        if (i % 2 === 1) tr.style.background = '#fafaf7';
        // FIX: bboxDet vem de m.detBox (caixa ruidosa) e bboxGT vem de m.gtBox (caixa real)
        var bboxDet = '(' + m.detBox.x + ',' + m.detBox.y + ',' + m.detBox.w + ',' + m.detBox.h + ')';
        var bboxGT = '(' + m.gtBox.x + ',' + m.gtBox.y + ',' + m.gtBox.w + ',' + m.gtBox.h + ')';
        var statusHtml = m.ok ? '<span style="color:#27ae60;font-weight:bold;">✔ True</span>' : '<span style="color:#e74c3c;font-weight:bold;">✖ False</span>';

        tr.innerHTML = 
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.id + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;"><b>' + m.classe + '</b></td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.area.toFixed(1) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.solidity.toFixed(3) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.vertices + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxDet + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + bboxGT + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + m.iou.toFixed(2) + '</td>' +
          '<td style="padding:5px 4px;border:1px solid #e4dcc8;text-align:center;">' + statusHtml + '</td>';
        elTbody.appendChild(tr);
      });
    }

    elToggleGT.addEventListener('click', function(){
      showGT = !showGT;
      if (showGT) {
        this.style.background = '#2563eb'; this.style.borderColor = '#1d4ed8'; this.style.color = '#ffffff';
        this.querySelector('span').textContent = '🟦';
      } else {
        this.style.background = '#f1ead7'; this.style.borderColor = '#d4cebe'; this.style.color = '#5e5a4a';
        this.querySelector('span').textContent = '⬜';
      }
      renderGrid();
    });

    elToggleDET.addEventListener('click', function(){
      showDET = !showDET;
      if (showDET) {
        this.style.background = '#059669'; this.style.borderColor = '#047857'; this.style.color = '#ffffff';
        this.querySelector('span').textContent = '🟩';
      } else {
        this.style.background = '#f1ead7'; this.style.borderColor = '#d4cebe'; this.style.color = '#5e5a4a';
        this.querySelector('span').textContent = '⬜';
      }
      renderGrid();
    });

    elBtnRand.addEventListener('click', gerarCenario);

    function setStage(s, btn){
      [root.querySelector('#ep0811_stage0'), root.querySelector('#ep0811_stage1')].forEach(function(b){
        b.style.background = 'transparent'; b.style.color = '#8a8371'; b.style.fontWeight = '600';
      });
      btn.style.background = '#26241d'; btn.style.color = '#fbf7ee'; btn.style.fontWeight = '700';
      stage = s; renderGrid();
    }

    root.querySelector('#ep0811_stage0').addEventListener('click', function(){ setStage(0, this); });
    root.querySelector('#ep0811_stage1').addEventListener('click', function(){ setStage(1, this); });

    gerarCenario();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0811');
    if (root) initSim(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.11:** Simulatore EP08_11: Classificazione Geometrica con Controlli Indipendenti di *Overlay BBox* (GT e DET)


In [ ]:
%%writefile EP08_11.py
# Codice Python

In [ ]:
TestSuite("EP08_11.py").run()

### EP08_12 🔴 Segmentazione di Istanza in Immagine Reale: Oggetti Geometrici

L'esempio classico di segmentazione di questo capitolo ha separato le "istanze" delle monete tramite disconnessione spaziale nella maschera binaria risultante dalla sogliatura di Otsu. In questo esercizio applicherai la stessa idea — ma ora su un'immagine reale con oggetti geometrici vari — concatenando pre-elaborazione, binarizzazione, estrazione dei contorni (`cv2.findContours`) e validazione del risultato rispetto a un ground truth di *bounding box*.

A differenza dell'esercizio precedente (etichettatura su maschera già pronta), qui parti dall'**immagine originale**: la qualità della tua segmentazione dipende direttamente dalle scelte di pre-elaborazione (filtraggio, sogliatura, operazioni morfologiche) effettuate prima di etichettare i componenti.

#### 📋 Linee Guida di Implementazione

1. **Input:** utilizzare l'immagine `00000.jpg`.
2. **Pre-elaborazione e segmentazione:** applicare le fasi necessarie (filtraggio, binarizzazione e operazioni morfologiche) per separare automaticamente gli oggetti dallo sfondo, senza ritagli manuali.
3. **Etichettatura e misurazione:** per ogni oggetto segmentato, determinare:
   - area;
   - centro di massa (centroide);
   - tipo, secondo il set `obj2`.
4. **Annotazione visiva:** scrivere, all'interno di ciascun oggetto, la sua area e la sigla del tipo (`obj2`).
5. **Validazione (IoU):** calcolare l'*Intersection over Union* (IoU) tra la *bounding box* rilevata (`cv2.boundingRect`) e la *bounding box* di ground truth del tipo corrispondente. Un oggetto è considerato correttamente segmentato solo se esiste esattamente una *bounding box* del tipo corretto con **IoU ≥ 0,5**.
6. **Output:** stampare, per ogni oggetto rilevato, il suo identificatore, il tipo e se è stato validato con successo (`acertou=1`) o meno. La stampa deve seguire l'ordine delle classi di `obj2` (0=Tria … 8=Cruz); all'interno della stessa classe, ordinare gli oggetti per coordinata verticale del centroide (`cy`) crescente. Alla fine, stampare l'accuratezza complessiva.

#### 📌 Vincoli Computazionali

* **Niente ritaglio manuale:** tutta la segmentazione deve essere eseguita sull'immagine completa.
* **Set di classi fisso:**
  ```python
  obj  = ['Triangulo','Quadrado','Pendagono','Hexagono','Heptagono','Circulo',
          'Elipse','Estrela','Cruz']
  obj2 = ['Tria','Quad','Pent','Hexa','Hept','Circ','Elip','Estr','Cruz']
  ```
* **Dimensione dell'immagine:** 608×608 pixel — utilizzata per denormalizzare le coordinate del file TXT.
* **Validazione tramite centro di massa:** un oggetto è considerato correttamente segmentato solo se il suo centroide si trova strettamente all'interno della *bounding box* di ground truth corrispondente allo stesso tipo di oggetto.

#### 🧠 Fondamenti Teorici

| Elemento | Ruolo nella segmentazione di istanza |
|---|---|
| Pre-elaborazione (filtraggio, sogliatura) | Fase che produce la maschera binaria a partire dall'immagine di intensità originale |
| `cv2.findContours` | Estrae i contorni dei componenti connessi nella maschera binaria |
| Momenti geometrici (`cv2.moments`) | Consentono di calcolare il centro di massa (centroide) di ogni contorno |
| `approxPolyDP` / vertici | Aiuta nella classificazione del tipo di oggetto (numero approssimativo di lati) |
| Validazione tramite *bounding box* | Conferma se l'istanza segmentata corrisponde spazialmente a un oggetto del ground truth, misurando l'accuratezza del metodo |

#### 📌 Esempio di Output Atteso

```
Objeto 1: tipo=Tria, validado=True
...
Acurácia: 88.89%
```

**Parametri fissi per la riproducibilità:** affinché l'output corrisponda al ground truth di correzione automatica, utilizza esattamente: filtro di area minima di 300 pixel; `cv2.approxPolyDP` con `epsilon = 0.02 * perimetro`; soglia di solidità 0.92 e conteggio dei vertici ≥ 9 (con ≥ 11 per distinguere Cruz da Estrela) per forme concave; rapporto d'aspetto 1.15 per distinguere Circulo da Elipse; soglia IoU = 0.5 nella validazione.

#### 📌 File di Riferimento (`.jpg` e `.txt`)

Per il debug locale, sono resi disponibili due file di riferimento (inclusi in questa consegna; quando li integri nel repository del capitolo, salvali in `all/cap08/dados/EP08/`):

* 📥 **Immagine (`00000.jpg`)**: immagine di oggetti geometrici utilizzata come input dell'esercizio. L'obiettivo è segmentare automaticamente ogni oggetto, determinarne il tipo e calcolarne le misure.
* 📥 **Ground Truth (`00000.txt`)**: file contenente le *bounding box* normalizzate degli oggetti presenti nell'immagine. Ogni riga contiene l'identificatore della classe e le coordinate normalizzate degli angoli superiore sinistro e inferiore destro, utilizzato per validare automaticamente la segmentazione.

La [Figura 8.12](#fig-08-ep12) presenta l'immagine di input e la stessa immagine con le *bounding box* disegnate a partire dal file di ground truth.

In [ ]:
import os
import urllib.request
from morph import mm

def garantir_e_baixar(nome):
    pasta = "dados/EP12"
    caminho = os.path.join(pasta, nome)

    os.makedirs(pasta, exist_ok=True)

    if not os.path.exists(caminho):
        url = (
            "https://raw.githubusercontent.com/"
            "fzampirolli/pdi-vc/master/all/cap08/dados/EP12/"
            + nome
        )
        print(f"Scaricamento {nome}...")
        urllib.request.urlretrieve(url, caminho)

    return caminho

img_arq = garantir_e_baixar("00000.jpg")
txt_arq = garantir_e_baixar("00000.txt")

img = mm.read(img_arq)
img_bb = mm.showBoundBox(img, txt_arq, fmt="yolo", show=False)

mm.show(
    [img, img_bb],
    titles=[
        "Immagine originale",
        "Bounding boxes del riferimento"
    ],
    cols=2,
    figsize=(10,5)
)

**Figura 8.12:** Simulatore EP08_12: Immagine utilizzata nell


In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0812" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧮 Simulatore EP08_12: Accuratezza della Segmentazione su Oggetti Multipli</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">🟢 indovinato se IoU ≥ soglia e tipo corretto</span>
  </div>


  <div style="padding:20px;background:white;overflow:auto">

    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Ogni forma ha un <i>boundbox</i> di riferimento (rettangolo tratteggiato, aderente alla forma) e un <i>boundbox</i> rilevato (rettangolo pieno, spostato/rumoroso). Regola rumore, bias e soglia IoU per vedere cambiare la validazione.
    </p>

    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;display:grid;grid-template-columns:1fr 1fr;gap:16px;">
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#c0392b;">rumore di segmentazione (px, jitter max per lato)</label><span id="ep0812_ruido_v" style="font-family:monospace;font-weight:bold;color:#c0392b;">0</span></div>
        <input id="ep0812_ruido" style="width:100%;accent-color:#c0392b;" max="20" min="0" step="1" type="range" value="0">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#2980b9;">bias sistematico in x (px)</label><span id="ep0812_bias_v" style="font-family:monospace;font-weight:bold;color:#2980b9;">0</span></div>
        <input id="ep0812_bias" style="width:100%;accent-color:#2980b9;" max="20" min="-20" step="1" type="range" value="0">
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;"><label style="font-size:12px;font-weight:bold;color:#27ae60;">soglia IoU</label><span id="ep0812_thr_v" style="font-family:monospace;font-weight:bold;color:#27ae60;">0.50</span></div>
        <input id="ep0812_thr" style="width:100%;accent-color:#27ae60;" max="0.9" min="0.1" step="0.05" type="range" value="0.5">
      </div>
      <div style="display:flex;align-items:center;gap:8px;">
        <input id="ep0812_erro" type="checkbox" style="accent-color:#8e44ad;width:16px;height:16px;">
        <label style="font-size:12px;font-weight:bold;color:#8e44ad;">simula errore di classificazione (2 oggetti con tipo scambiato)</label>
      </div>
    </div>

<div id="ep0812_svg" style="width:100%;max-width:420px;margin:0 auto 16px auto;"></div>

    <table style="width:100%;border-collapse:collapse;font-size:11px;font-family:monospace;margin-bottom:12px;">
      <thead>
        <tr style="background:#f3efe6;">
          <th style="padding:4px;border:1px solid #ddd;">id</th>
          <th style="padding:4px;border:1px solid #ddd;">tipo reale</th>
          <th style="padding:4px;border:1px solid #ddd;">tipo rilevato</th>
          <th style="padding:4px;border:1px solid #ddd;">IoU</th>
          <th style="padding:4px;border:1px solid #ddd;">≥ soglia</th>
          <th style="padding:4px;border:1px solid #ddd;">indovinato</th>
        </tr>
      </thead>
      <tbody id="ep0812_tbody"></tbody>
    </table>

    <div id="ep0812_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:12px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>

function svgNS(tag){
  var SVG_NS = "http" + "://www.w3.org/2000/svg";
  return document.createElementNS(SVG_NS, tag);
}

(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var ruidoEl = root.querySelector('#ep0812_ruido'), ruidovEl = root.querySelector('#ep0812_ruido_v');
    var biasEl = root.querySelector('#ep0812_bias'), biasvEl = root.querySelector('#ep0812_bias_v');
    var thrEl = root.querySelector('#ep0812_thr'), thrvEl = root.querySelector('#ep0812_thr_v');
    var erroEl = root.querySelector('#ep0812_erro');
    var svgWrap = root.querySelector('#ep0812_svg');
    var svgEl = svgNS('svg');
    svgEl.setAttribute('viewBox', '0 0 360 340');
    svgEl.setAttribute('style', 'width:100%;display:block;background:#0d0d0d;border-radius:12px;border:1px solid #333;');
    svgWrap.appendChild(svgEl);

    var tbody = root.querySelector('#ep0812_tbody');
    var dbg = root.querySelector('#ep0812_debug');

    var TIPOS = ['Circ','Tria','Quad','Cruz','Pent','Hexa','Hept','Estr','Elip'];
    var CORES = ['#a33a5b','#8fe3b0','#c98a7a','#5c4a5e','#3fbf5f','#d9c832','#e08a2b','#5c5470','#4a5a3a'];
    var FATOR = [1.0, 1.3, 0.7, 1.5, 0.9, 1.1, 0.6, 1.4, 0.8]; // sensibilidade individual ao ruído (fixa)
    var ANGS  = [30, 160, 260, 5, 200, 90, 340, 130, 240];      // direção fixa do erro por objeto (graus)
    var SCALE = [0.9, 1.15, 0.8, 1.2, 1.0, 0.95, 1.1, 1.25, 0.85]; // fator de encolhimento/expansão do bbox (fixo)

    var OBJS = [
      {cx:55,  cy:45,  r:22},
      {cx:150, cy:70,  r:24},
      {cx:250, cy:50,  r:22},
      {cx:320, cy:130, r:20},
      {cx:90,  cy:190, r:24},
      {cx:190, cy:220, r:24},
      {cx:275, cy:190, r:24},
      {cx:315, cy:270, r:22},
      {cx:150, cy:150, r:18}
    ];

    function svgShape(tipo, cx, cy, r, cor){
      var s = '';
      if(tipo==='Circ'){
        s = '<circle cx="'+cx+'" cy="'+cy+'" r="'+r+'" fill="'+cor+'"/>';
      } else if(tipo==='Elip'){
        s = '<ellipse cx="'+cx+'" cy="'+cy+'" rx="'+(r*1.1)+'" ry="'+(r*0.65)+'" fill="'+cor+'"/>';
      } else if(tipo==='Quad'){
        s = '<rect x="'+(cx-r*0.8)+'" y="'+(cy-r*0.8)+'" width="'+(r*1.6)+'" height="'+(r*1.6)+'" fill="'+cor+'"/>';
      } else if(tipo==='Cruz'){
        var w = r*0.5, l = r*1.4;
        s = '<g fill="'+cor+'">'+
            '<rect x="'+(cx-w/2)+'" y="'+(cy-l/2)+'" width="'+w+'" height="'+l+'"/>'+
            '<rect x="'+(cx-l/2)+'" y="'+(cy-w/2)+'" width="'+l+'" height="'+w+'"/></g>';
      } else {
        var sides = {Tria:3, Pent:5, Hexa:6, Hept:7}[tipo];
        var isStar = (tipo==='Estr');
        var pts = [];
        if(isStar){
          var spikes=5, outer=r, inner=r*0.45;
          for(var i=0;i<spikes*2;i++){
            var rad = (i%2===0)?outer:inner;
            var ang = Math.PI/spikes*i - Math.PI/2;
            pts.push((cx+rad*Math.cos(ang)).toFixed(1)+','+(cy+rad*Math.sin(ang)).toFixed(1));
          }
        } else {
          for(var i=0;i<sides;i++){
            var ang = 2*Math.PI/sides*i - Math.PI/2;
            pts.push((cx+r*Math.cos(ang)).toFixed(1)+','+(cy+r*Math.sin(ang)).toFixed(1));
          }
        }
        s = '<polygon points="'+pts.join(' ')+'" fill="'+cor+'"/>';
      }
      return s;
    }

    // bbox "verdadeiro": retângulo justo em torno da forma (sem folga artificial)
    function trueBBox(tipo, cx, cy, r){
      var hw = r, hh = r;
      if(tipo==='Elip'){ hw = r*1.1; hh = r*0.65; }
      else if(tipo==='Quad'){ hw = r*0.8; hh = r*0.8; }
      else if(tipo==='Cruz'){ hw = r*0.7; hh = r*0.7; }
      return [cx-hw, cy-hh, cx+hw, cy+hh];
    }

    function iou(a, b){
      var x1 = Math.max(a[0], b[0]), y1 = Math.max(a[1], b[1]);
      var x2 = Math.min(a[2], b[2]), y2 = Math.min(a[3], b[3]);
      var inter = Math.max(0, x2-x1) * Math.max(0, y2-y1);
      var areaA = (a[2]-a[0])*(a[3]-a[1]);
      var areaB = (b[2]-b[0])*(b[3]-b[1]);
      var uni = areaA + areaB - inter;
      return uni > 0 ? inter/uni : 0;
    }

    function render(){
      var ruido = parseInt(ruidoEl.value);
      var bias = parseInt(biasEl.value);
      var thr = parseFloat(thrEl.value);
      var erroAtivo = erroEl.checked;
      ruidovEl.textContent = ruido;
      biasvEl.textContent = bias;
      thrvEl.textContent = thr.toFixed(2);

      var svgContent = '';
      var rows = '';
      var acertos = 0;

      OBJS.forEach(function(o, i){
        var tipoReal = TIPOS[i];
        var cor = CORES[i];

        var gtBox = trueBBox(tipoReal, o.cx, o.cy, o.r);
        svgContent += '<g opacity="0.9">'+svgShape(tipoReal, o.cx, o.cy, o.r, cor)+'</g>';
        svgContent += '<rect x="'+gtBox[0]+'" y="'+gtBox[1]+'" width="'+(gtBox[2]-gtBox[0])+'" height="'+(gtBox[3]-gtBox[1])+'" fill="none" stroke="#aaa" stroke-dasharray="4,3" stroke-width="1.2"/>';

        // bbox detectado: escala fixa individual + jitter (ruído) + viés em x
        var scl = SCALE[i];
        var mag = ruido * FATOR[i];
        var ang = ANGS[i] * Math.PI/180;
        var jx = mag*Math.cos(ang), jy = mag*Math.sin(ang);
        var dw = (gtBox[2]-gtBox[0]) * scl, dh = (gtBox[3]-gtBox[1]) * scl;
        var dcx = o.cx + jx + bias, dcy = o.cy + jy;
        var detBox = [dcx-dw/2, dcy-dh/2, dcx+dw/2, dcy+dh/2];

        var val = iou(gtBox, detBox);
        var passaLimiar = val >= thr;

        var corDet = passaLimiar ? '#27ae60' : '#c0392b';
        svgContent += '<rect x="'+detBox[0]+'" y="'+detBox[1]+'" width="'+(detBox[2]-detBox[0])+'" height="'+(detBox[3]-detBox[1])+'" fill="none" stroke="'+corDet+'" stroke-width="1.6"/>';

        var tipoDetectado = tipoReal;
        if(erroAtivo && (i===1 || i===6)){
          tipoDetectado = TIPOS[(i+2)%TIPOS.length];
        }
        var tipoCorreto = (tipoDetectado === tipoReal);
        var acertou = passaLimiar && tipoCorreto;
        if(acertou) acertos++;

        svgContent += '<text x="'+(o.cx)+'" y="'+(gtBox[1]-6)+'" font-size="9" fill="#ccc" text-anchor="middle" font-family="monospace">'+ (i+1) +'</text>';

        rows += '<tr>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+(i+1)+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+tipoReal+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;'+(tipoCorreto?'':'color:#c0392b;font-weight:bold;')+'">'+tipoDetectado+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+val.toFixed(2)+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;">'+(passaLimiar?'sim':'não')+'</td>'+
          '<td style="padding:4px;border:1px solid #ddd;text-align:center;'+(acertou?'color:#27ae60;font-weight:bold;':'color:#c0392b;font-weight:bold;')+'">'+(acertou?'✔':'✘')+'</td>'+
          '</tr>';
      });

      svgEl.innerHTML = svgContent;
      tbody.innerHTML = rows;

      var acc = (acertos/OBJS.length*100).toFixed(1);
      dbg.textContent = 'Objetos validados: '+acertos+' / '+OBJS.length+'  →  Acurácia = '+acc+'%';
    }

    ruidoEl.addEventListener('input', render);
    biasEl.addEventListener('input', render);
    thrEl.addEventListener('input', render);
    erroEl.addEventListener('change', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0812');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 8.13:** Simulatore EP08_12: Accuratezza della Segmentazione su Multipli Oggetti (IoU)


In [ ]:
%%writefile EP08_12.py
# Codice Python

In [ ]:
TestSuite("EP08_12.py").run()